# Code to Reproduce Thesis Results

This notebook contains the code used to reproduce the results presented in the thesis: **"Smoothing the Curve: Forward Curve Construction and Factor Dynamics in German Electricity Futures"**

The notebook implements the complete empirical workflow described in the thesis, including data preparation, modelling and analysis, and the generation of figures and tables.

Results may differ slightly if different operating systems, software versions or package versions are used.

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker

from scipy.integrate import quad
from scipy import stats as sp_stats
from scipy.optimize import curve_fit

import re
import os
from datetime import datetime

import warnings
warnings.filterwarnings("ignore", "Covariance of the parameters")

The export flags below control which outputs are exported. It is possible to export either all or only selected results. To export all results generated in the thesis, set ```python export_all_results = True```. This automatically enables all individual export options. To export only specific outputs, keep ```python export_all_results = False``` and set the desired individual flags to `True`.

In [ ]:
# Master switch for exporting all results
export_all_results = True

# Switches for export of individual results
export_cleaned_data = False
export_maturity_horizons = False
export_seasonality_results = False
export_sample_curves = False
export_constructed_panels = False
export_panel_visualizations = False
export_desc_stats = False
export_pca_results = False
export_mkt_model_results = False

# If master switch is on, export all results
if export_all_results:
    export_cleaned_data = export_constructed_panels = export_panel_visualizations = export_desc_stats = True
    export_pca_results = export_maturity_horizons = export_seasonality_results = export_sample_curves = export_mkt_model_results = True

In [193]:
# Define the folders where the data and results are stored
DATA_DIR = "Data/"
RESULTS_DIR = "Results/"

# Create the results directory if it doesn't exist
os.makedirs(RESULTS_DIR, exist_ok=True)

In [194]:
# Define our subsamples to be used through the thesis
sample_start = p1_start = pd.Timestamp("2002-07-01")
p1_end = p2_start = pd.Timestamp("2007-01-01")
p2_end = p3_start = pd.Timestamp("2020-01-01")
sample_end = p3_end = pd.Timestamp("2025-01-01")

period_labels = {
    "full": "Full sample (Jul 2002 - Dec 2024)",
    "p1" : "Sub-period 1 (Jul 2002 - Dec 2006)",
    "p2" : "Sub-period 2 (Jan 2007 - Dec 2019)",
    "p3" : "Sub-period 3 (Jan 2020 - Dec 2024)",
}

period_bounds = {
    "full": (None, None),
    "p1": (p1_start, p1_end),
    "p2": (p2_start, p2_end),
    "p3": (p3_start, p3_end),
}

# 1. Data Loading and Cleaning

In [195]:
def load_data(filepath):
    """
    All the contracts data is stored in Excel files because we had access to Montel data only through their Excel add in.
    The data was downloaded to Excel in a messy format so we need to load all the contracts and their prices into a nice df.
    
    This function loads all data stored in multiple sheets from an Excel file into a single DataFrame.

    Each sheet has contracts laid out horizontally in blocks of 10 columns:
        col 0: Montel export details
        col 1-9: Open, High, Low, Settlement, Close, Volume, OpenInterest, Date, ContractName
    """

    xl_file = pd.ExcelFile(filepath)
    sheet_dfs = []

    # Loop through all sheets in the file
    for sheet_name in xl_file.sheet_names:
        df_raw = xl_file.parse(sheet_name, header=None)

        # Find column positions of 'Open' (always 1st col) to locate each contract block
        header_row = df_raw.iloc[0]
        open_positions = [i for i, v in enumerate(header_row) if v == "Open"]

        for open_pos in open_positions:
            # Extract the 9 data cols (Open to ContractName), skip Montel info col before Open
            block = df_raw.iloc[1:, open_pos : open_pos + 9].copy()
            block.columns = ["Open", "High", "Low", "Settlement", "Close", "Volume", "OpenInterest", "Date", "ContractName"]

            # Force ContractName to string (monthly contracts were stored as date in Excel for some reason)
            block["ContractName"] = block["ContractName"].apply(
                lambda x: x.strftime("%b-%y") if hasattr(x, "strftime") else
                        (str(x) if pd.notna(x) else None)
            )

            # Make sure Date col is datetime object and the rest are numeric
            block["Date"] = pd.to_datetime(block["Date"], errors="coerce")
            numeric_cols = ["Open", "High", "Low", "Settlement", "Close", "Volume", "OpenInterest"]
            for col in numeric_cols:
                block[col] = pd.to_numeric(block[col], errors="coerce")

            # Drop rows with missing values for settlment price, because contracts should have that the whole time they are traded
            block = block.dropna(subset=["Date", "ContractName", "Settlement"])

            # Reorder the columns
            block = block[["Date", "ContractName", "Open", "High", "Low", "Settlement", "Close", "Volume", "OpenInterest"]]

            # add the current contract block to the list
            if not block.empty:
                sheet_dfs.append(block)

    # Put all the contract blocks into single df and return that df
    result = (
        pd.concat(sheet_dfs, ignore_index=True)
        .drop_duplicates(subset=["Date", "ContractName"])
        .sort_values(["ContractName", "Date"])
        .reset_index(drop=True)
    )
    return result

# Use the load_data function on all our raw data files
df_weekly = load_data(DATA_DIR + "Montel_futures_raw_weekly.xlsx")
df_monthly = load_data(DATA_DIR + "Montel_futures_raw_monthly.xlsx")
df_quarterly = load_data(DATA_DIR + "Montel_futures_raw_quarterly.xlsx")
df_yearly = load_data(DATA_DIR + "Montel_futures_raw_yearly.xlsx")

# Quick summary to check the data
print(f"Weekly: {df_weekly.shape[0]:,} rows, {df_weekly['ContractName'].nunique()} contracts")
print(f"Monthly: {df_monthly.shape[0]:,} rows, {df_monthly['ContractName'].nunique()} contracts")
print(f"Quarterly: {df_quarterly.shape[0]:,} rows, {df_quarterly['ContractName'].nunique()} contracts")
print(f"Yearly: {df_yearly.shape[0]:,} rows, {df_yearly['ContractName'].nunique()} contracts")

# First date weekly contracts appear in the raw downloaded data (introduced sometime in 2017)
week_contracts_intro = df_weekly["Date"].min()
print(f"\nWeekly contracts first observed: {week_contracts_intro.date()}")

Weekly: 11,448 rows, 448 contracts
Monthly: 48,453 rows, 283 contracts
Quarterly: 46,589 rows, 94 contracts
Yearly: 31,065 rows, 24 contracts

Weekly contracts first observed: 2017-07-24


### Check that we have all the contracts

In [196]:
# -- Monthly -----------------------------------------------------------------
def check_monthly(df):
    # Find earliest and latest contract and check that each month between them has a contract too
    contracts = pd.to_datetime(sorted(df["ContractName"].unique()), format="%b-%y")
    expected = pd.date_range(contracts.min(), contracts.max(), freq="MS")
    missing = expected.difference(contracts)

    print("=== Monthly ===")
    print(f"Range: {contracts.min().strftime('%b %Y')} - {contracts.max().strftime('%b %Y')}")
    print(f"Expected: {len(expected)}   |   Found: {len(contracts)} |   Missing: {len(missing)}")
    if len(missing):
        print("Missing:", ", ".join(m.strftime("%b-%y") for m in missing))
    else:
        print("Complete - no gaps")

# -- Quarterly -----------------------------------------------------------------
def check_quarterly(df):
    contracts = sorted(df["ContractName"].unique())
    
    # Get the contract name as a tuple of format (year, quarter) e.g. (2026, 1)
    def parse_q(s):
        m = re.match(r"Q(\d)-(\d{4})", str(s))
        return (int(m.group(2)), int(m.group(1))) if m else None

    # Use the previous func to parse the contract names and sort them chronologically
    parsed = sorted(filter(None, [parse_q(c) for c in contracts]))
    if not parsed:
        print("Quarterly: no contracts with names in the expected format"); return

    # Get the earliest and latest quarter in our dataset
    (min_yr, min_q), (max_yr, max_q) = parsed[0], parsed[-1]

    # Generate every expected (year, quarter) pair in the earliest-latest range
    expected, actual = [], set(parsed)  # Actual is just unique list of all the paresed contract names
    yr, q = min_yr, min_q
    while (yr, q) <= (max_yr, max_q):
        expected.append((yr, q))
        q += 1
        if q > 4: q, yr = 1, yr + 1

    # Get list of missing contracts
    missing = [e for e in expected if e not in actual]

    print("\n=== Quarterly ===")
    print(f"Range: Q{min_q}-{min_yr}- Q{max_q}-{max_yr}")
    print(f"Expected: {len(expected)}   |   Found: {len(actual)}    |   Missing: {len(missing)}")
    if missing:
        print("Missing:", ", ".join(f"Q{q}-{yr}" for yr, q in missing))
    else:
        print("Complete - no gaps")

# -- Yearly -----------------------------------------------------------------
def check_yearly(df):
    contracts = sorted(df["ContractName"].unique())

    # Extract all year values from the contract names and order them
    years = sorted({int(re.search(r"\d{4}", c).group()) for c in contracts if re.search(r"\d{4}", c)})
    if not years:
        print("Yearly: no contracts with names in the expected format"); return

    # Find all the years from 1st to last year and check against the contracts we have
    expected = list(range(years[0], years[-1] + 1))
    missing = [y for y in expected if y not in years]

    print("\n=== Yearly ===")
    print(f"Range: Cal-{years[0]} - Cal-{years[-1]}")
    print(f"Expected: {len(expected)}  |  Found: {len(years)}  |  Missing: {len(missing)}")
    if missing:
        print("Missing:", ", ".join(f"Cal-{y}" for y in missing))
    else:
        print("Complete - no gaps")

# -- Weekly -----------------------------------------------------------------
def check_weekly(df):
    contracts = sorted(df["ContractName"].unique())

    # Parse the contract names to return actual dates of the Monday of that week
    def parse_wk(s):
        m = re.match(r"WK(\d+)-(\d{4})", str(s))
        if not m: return None
        wk, yr = int(m.group(1)), int(m.group(2))
        try:
            return datetime.strptime(f"{yr}-W{wk:02d}-1", "%G-W%V-%u")
        except ValueError:
            return None

    # Compare the actual parsed contract names with the range of earliest to latest weekly contract date
    parsed_map = {c: dt for c, dt in ((c, parse_wk(c)) for c in contracts) if dt is not None}
    actual_dates = pd.DatetimeIndex(sorted(parsed_map.values()))
    expected = pd.date_range(actual_dates.min(), actual_dates.max(), freq="W-MON")
    missing_dates = expected.difference(actual_dates)

    # Get the contract name back from the parsed value
    def fmt_wk(dt):
        iso = dt.isocalendar()
        return f"WK{iso[1]}-{iso[0]}"

    print("\n=== Weekly ===")
    print(f"Range: {min(parsed_map, key=parsed_map.get)} - {max(parsed_map, key=parsed_map.get)}")
    print(f"Expected: {len(expected)}   |   Found: {len(actual_dates)}  |   Missing: {len(missing_dates)}")
    if len(missing_dates):
        print("Missing:", ", ".join(fmt_wk(d) for d in missing_dates))
    else:
        print("Complete - no gaps")


check_monthly(df_monthly)
check_quarterly(df_quarterly)
check_yearly(df_yearly)
check_weekly(df_weekly)

=== Monthly ===
Range: Jul 2002 - Jan 2026
Expected: 283   |   Found: 283 |   Missing: 0
Complete - no gaps

=== Quarterly ===
Range: Q4-2002- Q1-2026
Expected: 94   |   Found: 94    |   Missing: 0
Complete - no gaps

=== Yearly ===
Range: Cal-2003 - Cal-2026
Expected: 24  |  Found: 24  |  Missing: 0
Complete - no gaps

=== Weekly ===
Range: WK30-2017 - WK8-2026
Expected: 448   |   Found: 448  |   Missing: 0
Complete - no gaps


### Limit the data to our full sample and check how many trading days there are

In [197]:
# Filter all dfs to keep only our full sample dates
for df in [df_monthly, df_weekly, df_quarterly, df_yearly]:
    mask = (df["Date"] >= sample_start) & (df["Date"] < sample_end)
    df.drop(index = df.index[~mask], inplace=True)

# Use yearly df as proxy to count the trading days cause yearly covers the full period
ref = df_yearly["Date"]

full_days = ref.nunique()
p1_days = ref[(ref >= p1_start) & (ref < p1_end)].nunique()
p2_days = ref[(ref >= p2_start) & (ref < p2_end)].nunique()
p3_days = ref[(ref >= p3_start) & (ref < p3_end)].nunique()  # 

print(f"{period_labels['full']}: {full_days:,} trading days")
print(f"{period_labels['p1']}: {p1_days:,} trading days")
print(f"{period_labels['p2']}: {p2_days:,} trading days")
print(f"{period_labels['p3']}: {p3_days:,} trading days")

Full sample (Jul 2002 - Dec 2024): 5,723 trading days
Sub-period 1 (Jul 2002 - Dec 2006): 1,142 trading days
Sub-period 2 (Jan 2007 - Dec 2019): 3,302 trading days
Sub-period 3 (Jan 2020 - Dec 2024): 1,279 trading days


### Add Delivery Period Columns and filter out dates where delivery is already in progress

In [198]:
def add_delivery_period(df, freq):
    """
    Function to add DeliveryStart, DeliveryEnd and DaysUntilDelivery cols.

    Delivery period conventions we used:
        weekly: Monday 00:00 - Sunday 24:00
        monthly: 1st of month - last day of month
        quarterly: 1st day of Q - last day of Q
        yearly: 1st Jan - 31st Dec
    """

    # Get the delivery period of the yearly contract based on its name
    def parse_yearly(name):
        m = re.match(r"Cal-(\d{4})", str(name))
        # Return NaN Date if the name is not as expected
        if not m:
            return pd.NaT, pd.NaT

        # Delivery period = 1st Jan of the Year - 31st Dec of the Year
        year = int(m.group(1))
        return pd.Timestamp(year, 1, 1), pd.Timestamp(year, 12, 31)

    # Get the delivery period of the monthly contract based on its name
    def parse_monthly(name):
        # Parse the contract name to get the month and year
        try:
            start = pd.to_datetime(name, format="%b-%y")
        except Exception:
            return pd.NaT, pd.NaT
        
        # Find the 1st day of the next month to easily find last day of the current month (to fix 30 vs 31 days issue etc)
        next_month = (
            pd.Timestamp(start.year + 1, 1, 1) if start.month == 12
            else pd.Timestamp(start.year, start.month + 1, 1)
        )

        # Return delivery period: 1st of the month to last day of the month (get as 1st day of next month - 1 day)
        return start, next_month - pd.Timedelta(days=1)

    # Get the delivery period of the quarterly contract based on its name
    def parse_quarterly(name):
        m = re.match(r"Q(\d)-(\d{4})", str(name))

        # Return NaN Date if the name is not as expected
        if not m:
            return pd.NaT, pd.NaT
        
        # Get the start of the current and next quarter
        q, year = int(m.group(1)), int(m.group(2))
        start_month = (q - 1) * 3 + 1
        end_month = start_month + 3
        start = pd.Timestamp(year, start_month, 1)
        next_q = (
            pd.Timestamp(year + 1, 1, 1) if end_month > 12
            else pd.Timestamp(year, end_month, 1)
        )

        # Return delivery period: 1st day of quarter and last day of quarter (start of next quarter - 1 day)
        return start, next_q - pd.Timedelta(days=1)

    # Get the delivery period of the weekly contract based on its name
    def parse_weekly(name):
        m = re.match(r"WK(\d+)-(\d{4})", str(name))
        
        # Return NaN Date if the name is not as expected
        if not m:
            return pd.NaT, pd.NaT
        wk, yr = int(m.group(1)), int(m.group(2))

        # Get Monday of the current week as start and Sunday of the current week as end (Monday + 6 days)
        start = pd.Timestamp(datetime.strptime(f"{yr}-W{wk:02d}-1", "%G-W%V-%u"))
        return start, start + pd.Timedelta(days=6)

    # Map the umbrella func arg on contract type to the helper func and apply the correct helper on the df
    parsers = {
        "yearly": parse_yearly,
        "monthly": parse_monthly,
        "quarterly": parse_quarterly,
        "weekly": parse_weekly,
    }
    period_map = {c: parsers[freq](c) for c in df["ContractName"].unique()}

    # Add delivery to the df
    df = df.copy()
    df["DeliveryStart"] = df["ContractName"].map(lambda c: period_map[c][0])
    df["DeliveryEnd"] = df["ContractName"].map(lambda c: period_map[c][1])

    # Compute days until delivery
    df["DaysUntilDelivery"] = (df["DeliveryStart"] - df["Date"]).dt.days

    return df

# Apply the above function on all 4 dfs
df_yearly = add_delivery_period(df_yearly, "yearly")
df_monthly = add_delivery_period(df_monthly, "monthly")
df_quarterly = add_delivery_period(df_quarterly, "quarterly")
df_weekly = add_delivery_period(df_weekly, "weekly")

In [199]:
# Remove rows where delivery has already started
for df in [df_monthly, df_weekly, df_quarterly, df_yearly]:
    mask = df["DaysUntilDelivery"] > 0
    df.drop(index=df.index[~mask], inplace=True)

### Export clean dfs

In [200]:
if export_cleaned_data:
    # Dfeine the cleaned dfs file names
    _clean_exports = [
        (df_monthly, "Monthly_clean.xlsx"),
        (df_quarterly, "Quarterly_clean.xlsx"),
        (df_yearly, "Yearly_clean.xlsx"),
        (df_weekly, "Weekly_clean.xlsx"),
    ]

    # Export the cleaned dfs to Excel
    for _df, _fname in _clean_exports:
        _df.to_excel(DATA_DIR + _fname, index=False)
        print(f"Exported: {DATA_DIR}{_fname}")

Exported: Data/Monthly_clean.xlsx
Exported: Data/Quarterly_clean.xlsx
Exported: Data/Yearly_clean.xlsx
Exported: Data/Weekly_clean.xlsx


### Term structure analysis - effective maturity horizons from raw contract data

In [201]:
threshold = 0.8  # horizon k is valid if >= 80% of trading days have >= k contracts

def max_horizon(df):
    """
    Find the largest k such that at least threshold % of trading days have >= k distinct contracts avaliable.
    e.g. k = 5 for monthly contracts means that at least threashold % of trading days has 5 monthly contracts avaliable
    to trade so then max M we can use in the term structure will be M5. This essentially means monthly contracts 
    trade with high enogh liquidity 5 months before they dleivery period.
    """
    
    # Count distinct contracts (by delivery start) available on each trading day
    counts = df.groupby("Date")["DeliveryStart"].nunique().rename("n")
    n_days = len(counts)                                  
    max_obs = int(counts.max()) if n_days > 0 else 0             

    # Keep only k values where >= threshold of days have at least k contracts to find the max k
    valid = [k for k in range(1, max_obs + 1) if (counts >= k).mean() >= threshold]
    k_max = max(valid) if valid else 0
    return k_max

W_full = max_horizon(df_weekly)
M_full = max_horizon(df_monthly)
Q_full = max_horizon(df_quarterly)
Y_full = max_horizon(df_yearly)

M_p1 = max_horizon(df_monthly[(df_monthly['Date'] >= p1_start) & (df_monthly['Date'] < p1_end)])
Q_p1 = max_horizon(df_quarterly[(df_quarterly['Date'] >= p1_start) & (df_quarterly['Date'] < p1_end)])
Y_p1 = max_horizon(df_yearly[(df_yearly['Date'] >= p1_start) & (df_yearly['Date'] < p1_end)])

W_p2 = max_horizon(df_weekly[(df_weekly['Date'] >= p1_end) & (df_weekly['Date'] < p3_start)])
M_p2 = max_horizon(df_monthly[(df_monthly['Date'] >= p1_end) & (df_monthly['Date'] < p3_start)])
Q_p2 = max_horizon(df_quarterly[(df_quarterly['Date'] >= p1_end) & (df_quarterly['Date'] < p3_start)])
Y_p2 = max_horizon(df_yearly[(df_yearly['Date'] >= p1_end) & (df_yearly['Date'] < p3_start)])

W_p3 = max_horizon(df_weekly[(df_weekly['Date'] >= p3_start) & (df_weekly['Date'] < p3_end)])
M_p3 = max_horizon(df_monthly[(df_monthly['Date'] >= p3_start) & (df_monthly['Date'] < p3_end)])
Q_p3 = max_horizon(df_quarterly[(df_quarterly['Date'] >= p3_start) & (df_quarterly['Date'] < p3_end)])
Y_p3 = max_horizon(df_yearly[(df_yearly['Date'] >= p3_start) & (df_yearly['Date'] < p3_end)])

# Return k-availability as a DataFrame (rows = k, cols = sub-periods)
def availability_table(df_src, contract_prefix, df_ref=None):
    """
    Return k-availability across 4 samples as a DataFrame.
    df_ref: full trading-day universe used as denominator (e.g. df_yearly).
    Dates in df_ref where df_src has no contracts are counted as 0 contracts,
    so the percentages reflect share of ALL trading days, not just days the
    contract type already exists.
    """
    ref = df_ref if df_ref is not None else df_src

    def period_counts(date_universe):
        # Contracts per day in df_src; days absent from df_src get 0
        present = df_src.groupby("Date")["DeliveryStart"].nunique()
        return present.reindex(date_universe, fill_value=0)

    full_dates = ref["Date"].drop_duplicates()
    p1_dates = ref.loc[(ref["Date"] >= p1_start) & (ref["Date"] < p1_end), "Date"].drop_duplicates()
    p2_dates = ref.loc[(ref["Date"] >= p1_end) & (ref["Date"] < p3_start), "Date"].drop_duplicates()
    p3_dates = ref.loc[(ref["Date"] >= p3_start) & (ref["Date"] < p3_end), "Date"].drop_duplicates()

    samples = [
        ("full", period_counts(full_dates)),
        ("p1", period_counts(p1_dates)),
        ("p2", period_counts(p2_dates)),
        ("p3", period_counts(p3_dates)),
    ]
    max_k = max(int(c.max()) for _, c in samples if len(c) > 0)
    rows = []
    for k in range(1, max_k + 1):
        row = {"Contract": f"{contract_prefix}{k}"}
        for lbl, counts in samples:
            row[lbl] = round((counts >= k).mean() * 100, 1)
        rows.append(row)
    return pd.DataFrame(rows).set_index("Contract")

In [202]:
if export_maturity_horizons:
    # Make sheet 1: summary with max horizon per contract type and sub-period
    _ct_map = [
        ("Weekly", df_weekly, "W"),
        ("Monthly", df_monthly, "M"),
        ("Quarterly", df_quarterly, "Q"),
        ("Yearly", df_yearly, "Y"),
    ]
    _samples = [(period_labels[k], t0, t1) for k, (t0, t1) in period_bounds.items()]
    summary_rows = []
    for ct_label, df_ct, prefix in _ct_map:
        row = {"Contract Type": ct_label}
        for s_label, t0, t1 in _samples:
            df_sub = df_ct if t0 is None else df_ct[(df_ct["Date"] >= t0) & (df_ct["Date"] < t1)]
            k = max_horizon(df_sub)
            row[s_label] = f"{prefix}{k}" if k > 0 else "-"
        summary_rows.append(row)
    df_summary = pd.DataFrame(summary_rows).set_index("Contract Type")

    # Make sheets 2-5: availability % per each k, one sheet per contract type
    out = RESULTS_DIR + "maturity_horizons.xlsx"
    with pd.ExcelWriter(out) as writer:
        df_summary.to_excel(writer, sheet_name="Max_Horizons")
        for ct_label, df_ct, prefix in _ct_map:
            availability_table(df_ct, prefix, df_ref=df_yearly).to_excel(writer, sheet_name=f"Availability_{prefix}")
    print(f"Exported: {out}")

Exported: Results/maturity_horizons.xlsx


# 2. Constructing Smooth Forward Curves

### Seasonality function

In [203]:
# Load the spot prices
df_spot = pd.read_excel(DATA_DIR + "Montel_spot_raw.xlsx", usecols=["Base", "Date"])

# Filter the spot data to our sample period + clean the df
df_spot["Date"] = pd.to_datetime(df_spot["Date"], errors="coerce")
df_spot["Base"] = pd.to_numeric(df_spot["Base"], errors="coerce")
df_spot = (
    df_spot
    .loc[lambda d: (d["Date"] >= sample_start) & (d["Date"] < sample_end)]
    .sort_values("Date")
    .reset_index(drop=True)
    .rename(columns={"Base": "Spot Price (EUR/MWh)"})
    [["Date", "Spot Price (EUR/MWh)"]]
)

In [204]:
# Simple sample mean as a benchmark for the spot price level
print(f"Sample mean (benchmark): {df_spot['Spot Price (EUR/MWh)'].mean():.4f} EUR/MWh")

Sample mean (benchmark): 54.7472 EUR/MWh


Quick look at the spot price data to see if there are any patterns (trend, seasonality)

In [205]:
# Seasonality, expected to be wavy
roll30 = df_spot["Spot Price (EUR/MWh)"].rolling(60, center=True).mean() 
# Annual trend        
roll365 = df_spot["Spot Price (EUR/MWh)"].rolling(365, center=True).mean()       

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df_spot["Date"], df_spot["Spot Price (EUR/MWh)"],
        lw=0.4, color="steelblue", alpha=0.6, label="Daily spot price")
ax.plot(df_spot["Date"], roll365,
        lw=2, color="red", label="365-day rolling mean (trend)")
ax.plot(df_spot["Date"], roll30,
        lw=1, color="darkblue", label="60-day rolling mean")

ax.set_title("German spot price (EUR/MWh)")
ax.set_xlabel("Date")
ax.set_ylabel("EUR/MWh")
ax.legend()
ax.grid(True, alpha=0.3)
if export_seasonality_results:
        plt.savefig(RESULTS_DIR + "spot_rolling_avg.png", dpi=150, bbox_inches="tight")
        print(f"Exported: {RESULTS_DIR}spot_rolling_avg.png")
plt.close()

Exported: Results/spot_rolling_avg.png


In [206]:
# Plotting the subsamples
subsamples = [(period_labels[k], t0, t1) for k, (t0, t1) in period_bounds.items() if k != "full"]

fig, axes = plt.subplots(3, 1, figsize=(14, 10), constrained_layout=True)

for ax, (label, start, end) in zip(axes, subsamples):
        sub = df_spot[(df_spot["Date"] >= start) & (df_spot["Date"] < end)].copy()
        roll30_s  = sub["Spot Price (EUR/MWh)"].rolling(30, center=True).mean()
        roll365_s = sub["Spot Price (EUR/MWh)"].rolling(365, center=True).mean()

        ax.plot(sub["Date"], sub["Spot Price (EUR/MWh)"], 
                lw=0.4, color="steelblue", alpha=0.6, label="Daily spot price")
        ax.plot(sub["Date"], roll365_s, 
                lw=2, color="red", label="365-day rolling mean (trend)")
        ax.plot(sub["Date"], roll30_s, 
                lw=1, color="darkblue", label="30-day rolling mean (seasonality)")
        ax.set_title(f"{label} (EUR/MWh)")
        ax.set_ylabel("EUR/MWh")
        ax.grid(True, alpha=0.3)
        ax.legend()
        ax.xaxis.set_major_locator(mdates.YearLocator())

if export_seasonality_results:
        plt.savefig(RESULTS_DIR + "spot_rolling_avg_subperiods.png", dpi=150, bbox_inches="tight")
        print(f"Exported: {RESULTS_DIR}spot_rolling_avg_subperiods.png")
plt.close()

Exported: Results/spot_rolling_avg_subperiods.png


#### 1. Constant seasonality

In [207]:
# Shared Fourier regression setup (reused in both models)
t = (df_spot["Date"] - df_spot["Date"].iloc[0]).dt.days.values.astype(float)
p = df_spot["Spot Price (EUR/MWh)"].values        
K = 2  # number of Fourier pairs (cos and sin) to include
fourier = np.column_stack([
    f(2 * np.pi * k * t / 365.25)
    for k in range(1, K + 1)
    for f in (np.cos, np.sin)
])

# Model A: Fourier only (preferred - no overlap between seasonal terms)
X_f = np.column_stack([np.ones(len(t)), t, fourier])
beta_f, *_ = np.linalg.lstsq(X_f, p, rcond=None)
a_f, b_f = beta_f[0], beta_f[1]
res_f = p - X_f @ beta_f
level_constant = float(a_f + b_f * t.mean())
n = len(t)
r2_f = 1 - res_f.var() / p.var()
adj_r2_f = 1 - (1 - r2_f) * (n - 1) / (n - X_f.shape[1])

print("--- Fourier only ---")
print(f"Trend:         {b_f * 365.25:+.4f} EUR/MWh per year")
print(f"Level:         {level_constant:.4f} EUR/MWh")
print(f"Residual std:  {res_f.std():.4f} EUR/MWh")
print(f"R²:            {r2_f:.4f}")
print(f"Adj. R²:       {adj_r2_f:.4f}")

# Model B: Fourier + month dummies (comparison - dummies add little beyond Fourier alone)
months = df_spot["Date"].dt.month.values
M_dummy = np.column_stack([(months == m).astype(float) for m in range(2, 13)])

X_fm = np.column_stack([np.ones(len(t)), t, M_dummy, fourier])
beta_fm, *_ = np.linalg.lstsq(X_fm, p, rcond=None)
a_fm, b_fm = beta_fm[0], beta_fm[1]
res_fm = p - X_fm @ beta_fm
level_constant_dummies = float(a_fm + b_fm * t.mean())
r2_fm = 1 - res_fm.var() / p.var()
adj_r2_fm = 1 - (1 - r2_fm) * (n - 1) / (n - X_fm.shape[1])

print("\n--- Fourier + month dummies ---")
print(f"Trend:         {b_fm * 365.25:+.4f} EUR/MWh per year")
print(f"Level:         {level_constant_dummies:.4f} EUR/MWh")
print(f"Residual std:  {res_fm.std():.4f} EUR/MWh")
print(f"R²:            {r2_fm:.4f}")
print(f"Adj. R²:       {adj_r2_fm:.4f}")

--- Fourier only ---
Trend:         +3.2022 EUR/MWh per year
Level:         54.6400 EUR/MWh
Residual std:  51.5856 EUR/MWh
R²:            0.1485
Adj. R²:       0.1480

--- Fourier + month dummies ---
Trend:         +3.2002 EUR/MWh per year
Level:         49.5720 EUR/MWh
Residual std:  51.5060 EUR/MWh
R²:            0.1511
Adj. R²:       0.1495


In [208]:
# Plot the constant seasonality we just derived over the spot price to see how it fits
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(df_spot["Date"], df_spot["Spot Price (EUR/MWh)"],
        lw=0.4, color="steelblue", alpha=0.6, label="Daily spot price")
ax.axhline(level_constant, color="red", lw=1.5, 
        label=f"Constant level seasonality ({level_constant:.1f} EUR/MWh)")

ax.set_title("German spot price with constant seasonality level (EUR/MWh)")
ax.set_xlabel("Date")
ax.set_ylabel("EUR/MWh")
ax.legend()
ax.grid(True, alpha=0.3)

if export_seasonality_results:
        plt.savefig(RESULTS_DIR + "spot_vs_constant.png", dpi=150, bbox_inches="tight")
        print(f"Exported: {RESULTS_DIR}spot_vs_constant.png")
plt.close()

Exported: Results/spot_vs_constant.png


#### 2. Trig seasonality

In [209]:
# Average spot price by day-of-year (DOY 1-366) across all sample years
doy_avg = (
    df_spot
    .assign(doy=df_spot["Date"].dt.dayofyear)
    .groupby("doy")["Spot Price (EUR/MWh)"]
    .mean()
)
d = doy_avg.index.values.astype(float)
s = doy_avg.values

# Fit K=2 Fourier harmonics to DOY averages to extract the seasonal component
K_sel = 2
omega = 2 * np.pi / 365.25
X_k2 = np.column_stack(
    [np.ones(len(d))] +
    [f(2 * np.pi * k * d / 365.25) for k in range(1, K_sel + 1) for f in (np.cos, np.sin)]
)
beta_k2, *_ = np.linalg.lstsq(X_k2, s, rcond=None)

# Get coeffs and fit residuals
alpha_trig = float(beta_k2[0])
cos_sin_trig  = beta_k2[1:]
res_k2 = s - X_k2 @ beta_k2

# print summary
print(f"Level (alpha): {alpha_trig:.4f} EUR/MWh")
for k in range(1, K_sel + 1):
    A_k = beta_k2[2*k - 1]
    B_k = beta_k2[2*k]
    gamma_k = float(np.sqrt(A_k**2 + B_k**2))
    peak_k  = int(round(np.arctan2(-B_k, A_k) / (k * omega) % (365.25 / k)))
    print(f"Harmonic {k}:    amplitude={gamma_k:.4f} EUR/MWh, peak DOY~{peak_k}")
print(f"R²:            {1 - res_k2.var() / s.var():.4f}")

# Print full equation for the trig seasonality we have fitted
print("\nFitted equation:")
eq = f"  S(d) = {alpha_trig:.4f}"
for k in range(1, K_sel + 1):
    A_k = float(beta_k2[2*k - 1])
    B_k = float(beta_k2[2*k])
    sign_B = "+" if B_k >= 0 else "-"
    eq += (f"\n         {A_k:+.4f}\u00b7cos(2\u03c0\u00b7{k}\u00b7d/365.25)"
        f" {sign_B} {abs(B_k):.4f}\u00b7sin(2\u03c0\u00b7{k}\u00b7d/365.25)")
print(eq)

Level (alpha): 54.5909 EUR/MWh
Harmonic 1:    amplitude=7.6558 EUR/MWh, peak DOY~83
Harmonic 2:    amplitude=1.9874 EUR/MWh, peak DOY~145
R²:            0.3429

Fitted equation:
  S(d) = 54.5909
         +1.1499·cos(2π·1·d/365.25) - 7.5690·sin(2π·1·d/365.25)
         +0.5345·cos(2π·2·d/365.25) + 1.9142·sin(2π·2·d/365.25)


In [210]:
# Plot the DOY averages and the fitted trig seasonality
d_smooth = np.linspace(1, d.max(), 500)
X_smooth = np.column_stack(
    [np.ones(500)] +
    [f(2 * np.pi * k * d_smooth / 365.25) for k in range(1, K_sel + 1) for f in (np.cos, np.sin)]
)

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.bar(d, s, width=1, color="steelblue", alpha=0.5, label="Day-of-year average spot price")
ax.plot(d_smooth, X_smooth @ beta_k2, color="darkblue", lw=2, label=f"Fitted trigonometric seasonality (K={K_sel})")
ax.set_xlabel("Day of year")
ax.set_ylabel("EUR/MWh")
ax.set_title(f"Average spot price by day of year and trig seasonality fit")
ax.legend()
ax.grid(True, alpha=0.3)

if export_seasonality_results:
        plt.savefig(RESULTS_DIR + "trig_DOY_fit.png", dpi=150, bbox_inches="tight")
        print(f"Exported: {RESULTS_DIR}trig_DOY_fit.png")
plt.close()

Exported: Results/trig_DOY_fit.png


In [211]:
# Plot the fitted trig seasonality vs actual spot
doy_full  = df_spot["Date"].dt.dayofyear.values.astype(float)
X_full = np.column_stack(
        [np.ones(len(doy_full))] +
        [f(2 * np.pi * k * doy_full / 365.25) for k in range(1, K_sel + 1) for f in (np.cos, np.sin)]
)
trig_full = X_full @ beta_k2

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df_spot["Date"], df_spot["Spot Price (EUR/MWh)"],
        lw=0.4, color="steelblue", alpha=0.6, label="Daily spot price")
ax.plot(df_spot["Date"], trig_full,
        lw=1.5, color="darkblue", label=f"Fitted trigonometric seasonality (K={K_sel})")
ax.set_title(f"German spot price with trig seasonality (EUR/MWh)")
ax.set_xlabel("Date")
ax.set_ylabel("EUR/MWh")
ax.legend()
ax.grid(True, alpha=0.3)

if export_seasonality_results:
        plt.savefig(RESULTS_DIR + "spot_vs_trig.png", dpi=150, bbox_inches="tight")
        print(f"Exported: {RESULTS_DIR}spot_vs_trig.png")
plt.close()

Exported: Results/spot_vs_trig.png


In [212]:
# Comparing different K values
rows = []
n = len(d)

# For each K 1-10, fit the Fourier model and compute fit quality metrics
for K_cmp in range(1, 11):
    k = 1 + 2 * K_cmp  # intercept + K cos/sin pairs
    X_cmp = np.column_stack(
        [np.ones(n)] +
        [f(2 * np.pi * ki * d / 365.25) for ki in range(1, K_cmp + 1) for f in (np.cos, np.sin)]
    )
    b_cmp, *_ = np.linalg.lstsq(X_cmp, s, rcond=None)
    resid = s - X_cmp @ b_cmp
    rss = float(np.dot(resid, resid))
    tss = float(np.dot(s - s.mean(), s - s.mean()))

    # Compute the metrics
    r2 = 1 - rss / tss
    adj_r2 = 1 - (rss / (n - k)) / (tss / (n - 1))
    aic = n * np.log(rss / n) + 2 * k
    aicc = aic + 2 * k * (k + 1) / (n - k - 1)
    bic = n * np.log(rss / n) + k * np.log(n)

    rows.append({"K": K_cmp, "params": k, "R²": r2, "adj R²": adj_r2,
                "AIC": aic, "AICc": aicc, "BIC": bic})

df_sel = pd.DataFrame(rows).set_index("K")

# Export results
if export_seasonality_results:
    _out = RESULTS_DIR + "trig_seasonality_K_comparison.xlsx"
    df_sel.to_excel(_out)
    print(f"Exported: {_out}")

Exported: Results/trig_seasonality_K_comparison.xlsx


#### Seasonality term func for main algo

In [213]:
def seasonal_constant(level):
    """lambda(u) = level (flat seasonality)"""
    def _f(u):
        return level
    return _f

def seasonal_trig(a0, cos_sin_betas, K = 2, period_days=365.25):
    """
    K-harmonic Fourier seasonality.
    cos_sin_betas: [A1, B1, A2, B2, ..., AK, BK]
        lambda(u) = a0 + sum_k A_k*cos(k*2pi*u_days/T) + B_k*sin(k*2pi*u_days/T)
    """
    betas = np.asarray(cos_sin_betas, dtype=float)
    def _f(u):
        u_days = u * 365.25
        result = float(a0)
        for k in range(1, K + 1):
            A_k, B_k = betas[2*(k-1)], betas[2*(k-1)+1]
            result += (A_k * np.cos(k * 2*np.pi * u_days / period_days)
                     + B_k * np.sin(k * 2*np.pi * u_days / period_days))
        return result
    return _f

### Main algorithm

In [214]:
REF_DATE = pd.Timestamp('2002-01-01')

def to_years(d, ref=REF_DATE):
    """
    Convert date to decimal years from ref date. Ref date is set by default to just before our sample starts.
    We use 365.25 days per year.
    """
    if pd.isna(d):
        return np.nan
    return (pd.Timestamp(d) - ref).days / 365.25

In [215]:
# -- H matrix -----------------------------------------
def build_H(knots):
    """
    Block-diagonal smoothness matrix H for the quartic spline.
    epsilon is expressed in local coordinates: s = u - tau_{j-1}
    """
    # Initialize H matrix as filled with 0s, only the diagonal will be filled in as needed
    n = len(knots) - 1
    H = np.zeros((5 * n, 5 * n))

    for j in range(n):
        # Initialize delta and hj matrix as filled with 0s, only the upper left 3x3 will be filled
        d = knots[j + 1] - knots[j]
        hj = np.zeros((5, 5))

        # Build this particular hj block, leaving the last 2 cols and rows to be 0
        hj[0, 0] = 144 / 5 * d ** 5
        hj[0, 1] = hj[1, 0] = 18 * d ** 4
        hj[0, 2] = hj[2, 0] = 8 * d ** 3
        hj[1, 1] = 12 * d ** 3
        hj[1, 2] = hj[2, 1] = 6 * d ** 2
        hj[2, 2] = 4 * d

        # Put hj into its correct place on the diagonal of H
        H[5 * j:5 * j + 5, 5 * j:5 * j + 5] = hj
    return H

# -- Constraint matrix A and rhs b -----------------------------------------
def build_constraints(knots, tau_b_arr, tau_e_arr, F_adj):
    """
    Build the full constraint system (A, b) for the Langrange multiplier problem.
    Constraints use local coordinates s = u - tau_j (s in [0, Delta_j] per piece).
    Matrix A has 3(n-1) + m + 1 rows (see below the constraints belonging to all the rows) and 5n cols.
    Vector b has length 3(n-1) + m + 1

    Constraints (in order):
        1. C0 continuity at interior knots -> n-1 rows
        2. C1 continuity at interior knots -> n-1 rows     
        3. C2 continuity at interior knots -> n-1 rows       
        4. epsilon'(tau_n) = 0 (flat long end) -> 1 row      
        5. Observed futures price matching -> m rows                  

    Parameters
    ----------
    knots : ndarray (n+1,) -> sorted knot points in decimal years
    tau_b_arr : ndarray (m,) -> delivery-period start times
    tau_e_arr : ndarray (m,) -> delivery-period end times
    F_adj : ndarray (m,) -> F_i*(tau_e - tau_b) - integral lambda du (rhs for the price constraint)
    """

    # Initialize the matrix A and vector b with their assigned dimensions
    n = len(knots) - 1
    m = len(tau_b_arr)
    n_con = 3 * (n - 1) + 1 + m

    A = np.zeros((n_con, 5 * n))
    b = np.zeros(n_con)
    row = 0

    # -- C0: a_j delta^4 + b_j delta^3 + c_j delta^2 + d_j delta + e_j = e_{j+1} 
    for j in range(n - 1):
        d = knots[j + 1] - knots[j]
        A[row, 5 * j: 5 * j + 5] = [d ** 4, d ** 3, d ** 2, d, 1.0]
        A[row, 5 * (j + 1) + 4]  = -1.0
        row += 1

    # -- C1: 4a_j delta^3 + 3b_j delta^2 + 2c_j delta + d_j = d_{j+1} 
    for j in range(n - 1):
        d = knots[j + 1] - knots[j]
        A[row, 5 * j: 5 * j + 4] = [4 * d ** 3, 3 * d ** 2, 2 * d, 1.0]
        A[row, 5 * (j + 1) + 3]  = -1.0
        row += 1

    # -- C2: 12a_j delta^2 + 6b_j delta + 2c_j = 2c_{j+1} 
    for j in range(n - 1):
        d = knots[j + 1] - knots[j]
        A[row, 5 * j: 5 * j + 3] = [12 * d ** 2, 6 * d, 2.0]
        A[row, 5 * (j + 1) + 2]  = -2.0
        row += 1

    # -- Endpoint: 4a_n delta_n^3 + 3b_n delta_n^2 + 2c_n delta_n + d_n = 0 
    j = n - 1
    d = knots[n] - knots[n - 1]
    A[row, 5 * j: 5 * j + 4] = [4 * d ** 3, 3 * d ** 2, 2 * d, 1.0]
    row += 1

    # -- Observed futures price: integral_{tau_b}^{tau_e} epsilon(u) du = F_adj_i 
    # Since ALL tau_b/tau_e are knot points, delivery period i covers pieces
    # k_b, k_b+1, …, k_e - 1 exactly (no partial pieces).
    knot_map = {round(v, 10): idx for idx, v in enumerate(knots)}

    for i in range(m):
        k_b = knot_map[round(tau_b_arr[i], 10)]
        k_e = knot_map[round(tau_e_arr[i], 10)]
        for j in range(k_b, k_e):
            d = knots[j + 1] - knots[j]
            # integral_0^d (a s^4 + b s^3 + c s^2 + d s + e) ds
            A[row, 5 * j: 5 * j + 5] = [d**5 / 5, d**4 / 4, d**3 / 3, d**2 / 2, d]
        b[row] = F_adj[i]
        row += 1

    return A, b

# -- Lagrange multiplier solver using QR factorisation --------------------------------------
def solve_Lagrange(H, A, b_vec):
    """
    Solve following system of linear equations:
        [2H  A^T] [x]   [0]
        [A    0 ] [l] = [b]
    using QR factorisation.
    Returns x reshaped as (n, 5) coefficient array [a, b, c, d, e] per piece.
    """

    n5 = H.shape[0]
    nc = A.shape[0]
    K = np.zeros((n5 + nc, n5 + nc))
    K[:n5, :n5] = 2 * H
    K[:n5, n5:] = A.T
    K[n5:, :n5] = A
    rhs = np.zeros(n5 + nc)
    rhs[n5:] = b_vec
    Q, R = np.linalg.qr(K)
    sol = np.linalg.solve(R, Q.T @ rhs)
    return sol[:n5].reshape(-1, 5)

In [216]:
# -- Main fitting function ---------------------------------------------------
def fit_forward_curve(tau_b_arr, tau_e_arr, prices, seasonal_func = None):
    """
    Fit the maximum-smoothness forward curve for a single trading day.

    Parameters
    ----------
    tau_b_arr : array-like delivery-period start times
    tau_e_arr : array-like delivery-period end times
    prices : array-like observed futures prices F_i (EUR/MWh)
    seasonal_func : seasonality function, default is None which coresponds to 0 seasonality

    Returns
    -------
    coeffs : ndarray (n, 5) polynomial [a,b,c,d,e] in local coords per piece
    knots : ndarray (n+1,) knot grid in decimal years
    success : bool
    """

    # Make sure inputs good
    tau_b = np.asarray(tau_b_arr, dtype=float)
    tau_e = np.asarray(tau_e_arr, dtype=float)
    F = np.asarray(prices, dtype=float)

    mask = np.isfinite(F) & np.isfinite(tau_b) & np.isfinite(tau_e) & (tau_e > tau_b)
    tau_b, tau_e, F = tau_b[mask], tau_e[mask], F[mask]

    # If there are less than 2 observed contracts on the trading day, the optimization fails
    if len(F) < 2:
        return None, None, False

    # Build knot grid from all unique delivery-period boundaries
    knots = np.array(sorted(set(np.round(tau_b, 10)) | set(np.round(tau_e, 10))))
    n = len(knots) - 1
    # Optimization fails if all contracts on the day have the same delivery period
    if n < 1:
        return None, None, False

    # Add the seasonality if specified, otherwise use default 0 seasonality
    if seasonal_func is not None:
        lam_int = np.array([quad(seasonal_func, tau_b[i], tau_e[i])[0]
                            for i in range(len(F))])
    else:
        lam_int = np.zeros(len(F))

    F_adj = F * (tau_e - tau_b) - lam_int

    try:
        # Get the fitted spline coefs using the funcs we have already built
        H_mat = build_H(knots)
        A_mat, b_vec = build_constraints(knots, tau_b, tau_e, F_adj)
        coeffs = solve_Lagrange(H_mat, A_mat, b_vec)

        # Quick sanity check - evaluate curve at mid-points
        mid = (knots[:-1] + knots[1:]) / 2
        vals = np.array([
            coeffs[j, 0]*s**4 + coeffs[j, 1]*s**3 + coeffs[j, 2]*s**2
            + coeffs[j, 3]*s + coeffs[j, 4]
            for j, s in enumerate(mid - knots[:-1])])

        # Return the coefs if everything ok
        if not np.all(np.isfinite(vals)):
            return None, None, False
        return coeffs, knots, True
    
    except Exception as exc:
        warnings.warn(f"fit_forward_curve failed: {exc}")
        return None, None, False

In [217]:
# -- Constructing the forward curve --------------------------------------
def eval_forward(coeffs, knots, u_arr, seasonal_func=None):
    """
    Construct the forward curve by evaluating f(u) = lambda(u) + epsilon(u) 
    at every point in u_arr (decimal years).
    """
    u_arr = np.atleast_1d(np.asarray(u_arr, dtype=float))
    n = len(knots) - 1
    result = np.full(u_arr.shape, np.nan)

    # For each point evaluate the epsilon(u) and add lambda(u)
    for k, u in enumerate(u_arr):
        if u < knots[0] - 1e-9 or u > knots[-1] + 1e-9:
            continue
        j = min(n - 1, max(0, np.searchsorted(knots, u, side='right') - 1))
        s = u - knots[j]
        a, b, c, d, e = coeffs[j]
        eps = a*s**4 + b*s**3 + c*s**2 + d*s + e
        lam = seasonal_func(u) if seasonal_func is not None else 0.0
        result[k] = lam + eps
    return result

# -- Getting the synthetic futures prices --------------------------------------
def synthetic_futures_price(coeffs, knots, tau_b, tau_e, seasonal_func=None):
    """
    Compute the synthetic futures price for delivery period [tau_b, tau_e] as the
    time-average of the fitted forward curve:
        F(tau_b, tau_e) = (1/Delta) * integral_{tau_b}^{tau_e} f(u) du

    The spline part is integrated analytically segment by segment using the
    antiderivative of the quartic polynomial. The seasonal component lambda(u)
    is integrated numerically via quad.
    """
    # Check
    if coeffs is None:
        return np.nan

    # Make sure inputs are correct + initialize
    tau_b, tau_e = float(tau_b), float(tau_e)
    total = 0.0

    # Integrate epsilon(u) analytically over each segment's overlap with [tau_b, tau_e]
    for j in range(len(knots) - 1):
        s0 = max(tau_b, knots[j]) - knots[j]
        s1 = min(tau_e, knots[j + 1]) - knots[j]
        if s1 <= s0 + 1e-12:
            continue
        a, b, c, d, e = coeffs[j]
        def _I(s):
            return a*s**5/5 + b*s**4/4 + c*s**3/3 + d*s**2/2 + e*s
        total += _I(s1) - _I(s0)

    # Add seasonal integral numerically if seasonality is specified
    if seasonal_func is not None:
        lam_int, _ = quad(seasonal_func, tau_b, tau_e)
        total += lam_int
    
    # Return time-average price F = (1/Delta) * integral, or nan if delivery period is degenerate
    delta = tau_e - tau_b
    return total / delta if delta > 1e-10 else np.nan

### Data Preparation

In [218]:
def prepare_for_fitting(df):
    """
    Convert delivery boundaries to decimal years (tau_b, tau_e).
    DeliveryEnd is stored as last-day-at-midnight therefore we add +1 day to make tau_e
    the first moment after the delivery period ends.
    """
    out = df.copy()
    out["tau_b"] = out["DeliveryStart"].map(to_years)
    out["tau_e"] = out["DeliveryEnd"].map(lambda d: to_years(d + pd.Timedelta(days=1)))
    return out.rename(columns={"Date": "date"})[["date", "ContractName", "Settlement", "tau_b", "tau_e"]]

# Use the func above on all the datasets
df_m_fit = prepare_for_fitting(df_monthly)
df_q_fit = prepare_for_fitting(df_quarterly)
df_y_fit = prepare_for_fitting(df_yearly)
df_w_fit = prepare_for_fitting(df_weekly)

# Merge all sources into one dataframe
df_all_fit = pd.concat([df_m_fit, df_q_fit, df_y_fit, df_w_fit], ignore_index=True)

# Deduplicate (tau_b, tau_e) pairs on the same day
df_all_fit = (
    df_all_fit.sort_values(["date", "tau_b", "tau_e"])
    .drop_duplicates(subset=["date", "tau_b", "tau_e"], keep="first")
)

# Collect valid contracts per trading day (at least 2 contracts, no missing values, positive delivery span)
daily_contracts = {}
for date, grp in df_all_fit.groupby("date"):
    g = grp.dropna(subset=["tau_b", "tau_e", "Settlement"])
    g = g[g["tau_e"] > g["tau_b"] + 1e-6]
    if len(g) >= 2:
        daily_contracts[date] = {
            "tau_b": g["tau_b"].values,
            "tau_e": g["tau_e"].values,
            "prices": g["Settlement"].values,
        }

all_dates = sorted(daily_contracts.keys())

print(f"Trading days with ≥2 contracts: {len(all_dates):,}")
print(f"Date range: {all_dates[0].date()} - {all_dates[-1].date()}")
n_contracts = [len(v["prices"]) for v in daily_contracts.values()]
print(f"Contracts/day: median {np.median(n_contracts):.0f}, min {min(n_contracts)}, max {max(n_contracts)}")

Trading days with ≥2 contracts: 5,723
Date range: 2002-07-01 - 2024-12-31
Contracts/day: median 21, min 13, max 29


### Fit Forward Curves for All Trading Days

In [219]:
specs = {
    "trig": seasonal_trig(alpha_trig, cos_sin_trig),    # our own trig seasonality based on spot
    "constant": seasonal_constant(level_constant),      # constant level based on spot
    #"zero": None,   # 0 seasonality
}

curve_stores = {}
failed_by_spec = {}

# For each seasonality spec, fit the forward curve for every trading day and store the results
for spec_name, seasonal_func in specs.items():
    print(f"\n── Fitting spec: {spec_name!r} ({len(all_dates):,} days) ──", flush=True)
    store  = {}
    failed = []
    report_every = max(1, len(all_dates) // 20)

    for i, date in enumerate(all_dates):
        d = daily_contracts[date]
        coeffs, knots, ok = fit_forward_curve(
            d["tau_b"], d["tau_e"], d["prices"],
            seasonal_func=seasonal_func,
        )
        if ok:
            store[date] = {"coeffs": coeffs, "knots": knots}
        else:
            failed.append(date)

        if (i + 1) % report_every == 0 or i == len(all_dates) - 1:
            pct = (i + 1) / len(all_dates) * 100
            print(f"  {i+1}/{len(all_dates)} ({pct:5.1f}%)  fitted: {len(store):,}  failed: {len(failed)}", flush=True)

    # Store successfull and failed fits
    curve_stores[spec_name] = store
    failed_by_spec[spec_name] = failed
    print(f"  Done. Fitted: {len(store):,} / {len(all_dates):,}  Failed: {len(failed)}")


── Fitting spec: 'trig' (5,723 days) ──
  286/5723 (  5.0%)  fitted: 286  failed: 0
  572/5723 ( 10.0%)  fitted: 572  failed: 0
  858/5723 ( 15.0%)  fitted: 858  failed: 0
  1144/5723 ( 20.0%)  fitted: 1,144  failed: 0
  1430/5723 ( 25.0%)  fitted: 1,430  failed: 0
  1716/5723 ( 30.0%)  fitted: 1,716  failed: 0
  2002/5723 ( 35.0%)  fitted: 2,002  failed: 0
  2288/5723 ( 40.0%)  fitted: 2,288  failed: 0
  2574/5723 ( 45.0%)  fitted: 2,574  failed: 0
  2860/5723 ( 50.0%)  fitted: 2,860  failed: 0
  3146/5723 ( 55.0%)  fitted: 3,146  failed: 0
  3432/5723 ( 60.0%)  fitted: 3,432  failed: 0
  3718/5723 ( 65.0%)  fitted: 3,718  failed: 0
  4004/5723 ( 70.0%)  fitted: 4,004  failed: 0
  4290/5723 ( 75.0%)  fitted: 4,290  failed: 0
  4576/5723 ( 80.0%)  fitted: 4,576  failed: 0
  4862/5723 ( 85.0%)  fitted: 4,862  failed: 0
  5148/5723 ( 90.0%)  fitted: 5,148  failed: 0
  5434/5723 ( 95.0%)  fitted: 5,434  failed: 0
  5720/5723 ( 99.9%)  fitted: 5,720  failed: 0
  5723/5723 (100.0%)  fitted

### Sample Curve Visualisations and Analysis

Sample Curve Plots

In [220]:
# Pick what kind of curve(s) you want to plot (dates, seasonality)
# seasonality can be ["trig"], ["constant"], or both
plot_dates = [
    (pd.Timestamp("2022-10-31"), ["trig"]),
    (pd.Timestamp("2023-08-02"), ["trig"]),
    (pd.Timestamp("2008-08-05"), ["trig", "constant"]),
]

spec_styles = {
    "trig": ("darkblue", "-",  "Trigonometric seasonality"),
    "constant": ("red", "--", "Constant level seasonality"),
}

# Plot and save 1 fig per date
for target_date, plot_specs in plot_dates:
    # If the picked date is not a trading day, print dates around that are avaliable
    if target_date not in curve_stores["trig"]:
        nearby = [d for d in sorted(curve_stores["trig"]) if abs((d - target_date).days) <= 5]
        print(f"No fitted curve for {target_date.date()}. Available nearby dates: {nearby}")
        continue
    d_t = daily_contracts[target_date]

    # Compute max price-matching error per seasonality (used in legend labels)
    max_errors = {}
    for spec_name in plot_specs:
        cs_s = curve_stores[spec_name][target_date]
        f_c  = np.array([
            synthetic_futures_price(cs_s["coeffs"], cs_s["knots"], tb, te, seasonal_func=specs[spec_name])
            for tb, te in zip(d_t["tau_b"], d_t["tau_e"])
        ])
        max_errors[spec_name] = np.nanmax(np.abs(f_c - d_t['prices']))

    fig, ax = plt.subplots(figsize=(9, 4.5))
    ax.set_title(f"Forward curve on {target_date.date()} ({', '.join(plot_specs)})")

    # Observed futures prices
    for tb, te, price in zip(d_t["tau_b"], d_t["tau_e"], d_t["prices"]):
        d_start = REF_DATE + pd.Timedelta(days=tb * 365.25)
        d_end = REF_DATE + pd.Timedelta(days=te * 365.25)
        ax.hlines(price, d_start, d_end, colors="hotpink", lw=2.5, alpha=0.7)
    ax.hlines([], [], [], colors="hotpink", lw=2.5, label="Observed futures prices")

    # Fitted curves
    for spec_name in plot_specs:
        col, ls, lbl = spec_styles[spec_name]
        cs_s = curve_stores[spec_name][target_date]
        u_s = np.linspace(cs_s["knots"][0], cs_s["knots"][-1], 2000)
        f_s = eval_forward(cs_s["coeffs"], cs_s["knots"], u_s, seasonal_func=specs[spec_name])
        dates_s = [REF_DATE + pd.Timedelta(days=u * 365.25) for u in u_s]
        _leg_lbl = f"{lbl}  (max err: {max_errors[spec_name]:.4f})" if len(plot_specs) > 1 else lbl
        ax.plot(dates_s, f_s, lw=1.8, color=col, ls=ls, label=_leg_lbl)

    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
    ax.set_xlabel("Delivery date")
    ax.set_ylabel("EUR/MWh")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()

    spec_tag = "_".join(plot_specs)
    fname = f"example_curve_{spec_tag}_{target_date.date()}.png"
    
    if export_sample_curves:
        plt.savefig(RESULTS_DIR + fname, dpi=150, bbox_inches="tight")
        print(f"Exported: {RESULTS_DIR}{fname}")
        plt.close()

Exported: Results/example_curve_trig_2022-10-31.png
Exported: Results/example_curve_trig_2023-08-02.png
Exported: Results/example_curve_trig_constant_2008-08-05.png


Negative Curves

In [221]:
# Check for negative values across all fitted curves
N_GRID = 500  # points per curve

# Evaluate each curve on the grid and flag any date where f(u) < 0
for spec in ["trig", "constant"]:
    neg_dates = []
    min_vals = []

    for date, cs in curve_stores[spec].items():
        u = np.linspace(cs["knots"][0], cs["knots"][-1], N_GRID)
        f = eval_forward(cs["coeffs"], cs["knots"], u, seasonal_func=specs[spec])
        min_f = float(np.nanmin(f))
        if min_f < 0:
            neg_dates.append(date)
            min_vals.append(min_f)

    print(f"\nSpec: {spec}")
    print(f"  Total curves   : {len(curve_stores[spec])}")
    print(f"  Negative curves: {len(neg_dates)}")
    if neg_dates:
        overall_min = min(min_vals)
        print(f"  Min value found: {overall_min:.4f} EUR/MWh")
        print(f"  Negative Dates : {[str(d.date()) for d in sorted(neg_dates)]}")
    else:
        print("  All curves non-negative.")


Spec: trig
  Total curves   : 5723
  Negative curves: 5
  Min value found: -786.9887 EUR/MWh
  Negative Dates : ['2018-05-25', '2020-04-29', '2022-10-31', '2022-12-29', '2024-12-27']

Spec: constant
  Total curves   : 5723
  Negative curves: 5
  Min value found: -767.9969 EUR/MWh
  Negative Dates : ['2018-05-25', '2020-04-29', '2022-10-31', '2022-12-29', '2024-12-27']


Seasonality Choice Differences

In [222]:
# Calculate distance between trig and constant forward curves
N_GRID = 300
common_dates = sorted(set(curve_stores["trig"]) & set(curve_stores["constant"]))

# For each trading day: d_t = RMSD over a fine maturity grid
records = []
for date in common_dates:
    cs_tr = curve_stores["trig"][date]
    cs_co = curve_stores["constant"][date]
    u_lo = max(cs_tr["knots"][0],  cs_co["knots"][0])
    u_hi = min(cs_tr["knots"][-1], cs_co["knots"][-1])
    if u_hi <= u_lo:
        continue
    u = np.linspace(u_lo, u_hi, N_GRID)
    f_tr = eval_forward(cs_tr["coeffs"], cs_tr["knots"], u, seasonal_func=specs["trig"])
    f_co = eval_forward(cs_co["coeffs"], cs_co["knots"], u, seasonal_func=specs["constant"])
    diff = f_tr - f_co
    d_abs = np.sqrt(np.nanmean(diff**2))
    d_rel = d_abs / np.sqrt(np.nanmean(f_tr**2))
    records.append({"date": date, "d_abs": d_abs, "d_rel": d_rel})

df_dist = pd.DataFrame(records).set_index("date").sort_index()

# Summary table (mean, median, P10, P90, max) of the distance
rows = []
for period, (t0, t1) in period_bounds.items():
    sub = df_dist if t0 is None else df_dist[(df_dist.index >= t0) & (df_dist.index < t1)]
    if sub.empty:
        continue
    for col, label, scale in [("d_abs", "Abs RMSD (EUR/MWh)", 1), ("d_rel", "Rel RMSD (%)", 100)]:
        s = sub[col] * scale
        rows.append({
            "Period": period_labels[period],
            "Metric": label,
            "Mean": round(s.mean(), 3),
            "Median": round(s.median(), 3),
            "P10": round(s.quantile(0.10), 3),
            "P90": round(s.quantile(0.90), 3),
            "Max": round(s.max(), 3),
        })
df_summary = pd.DataFrame(rows)

# Time-series plot of the distance
fig, axes = plt.subplots(2, 1, figsize=(9, 4.5), sharex=True)

period_shading = [
    ("p1", p1_start, p1_end, "steelblue"),
    ("p2", p2_start, p2_end, "darkorange"),
    ("p3", p3_start, sample_end,"seagreen"),
]

for ax, col, ylabel in zip(axes, ["d_abs", "d_rel"], ["RMSD  (EUR/MWh)", "Relative RMSD (%)"]):
    vals = df_dist[col] * (100 if col == "d_rel" else 1)
    ax.plot(df_dist.index, vals, lw=0.7, color="k", alpha=0.8)
    for pid, t0, t1, col_shade in period_shading:
        ax.axvspan(t0, t1, alpha=0.15, color=col_shade, label=period_labels[pid].split("(")[0].strip())
    ax.set_ylabel(ylabel, fontsize=12)
    ax.grid(True, alpha=0.25)

axes[1].legend(fontsize=10, loc="upper right")
axes[0].set_title("Forward-curve distance: Trigonometric vs Constant seasonality", fontsize=12)
axes[1].set_xlabel("Date", fontsize=12)
plt.tight_layout()

# Export both summary tbl and plot
if export_sample_curves:
    plt.savefig(RESULTS_DIR + "curve_dist_trig_vs_const.png", dpi=150, bbox_inches="tight")
    print(f"Exported: {RESULTS_DIR}curve_dist_trig_vs_const.png")
    df_summary.to_excel(RESULTS_DIR + "curve_dist_summary.xlsx", index=False)
    print(f"Exported: {RESULTS_DIR}curve_dist_summary.xlsx")
plt.close()

Exported: Results/curve_dist_trig_vs_const.png
Exported: Results/curve_dist_summary.xlsx


# 3. Panel Construction With The Synthetic Contracts

For each trading date we derive "n-th period ahead" prices:
- W1–W4 : next 4 ISO weeks (Mon–Sun)
- M1–M6 : next 6 calendar months
- Q1–Q8 : next 8 calendar quarters
- Y1–Y3 : next 3 calendar years

In [223]:
def next_monday(date):
    """Return the Monday of the week starting after 'date'."""
    # ISO weekday: Mon=1 ... Sun=7
    days_ahead = 7 - date.weekday() 
    return (date + pd.Timedelta(days=days_ahead)).normalize()

def first_of_next_month(date):
    """Return the 1st day of the next month."""
    if date.month == 12:
        return pd.Timestamp(date.year + 1, 1, 1)
    return pd.Timestamp(date.year, date.month + 1, 1)

def first_of_next_quarter(date):
    """Return the 1st day of next quarter."""
    q = (date.month - 1) // 3            
    start_month = (q + 1) * 3 + 1    
    if start_month > 12:
        return pd.Timestamp(date.year + 1, 1, 1)
    return pd.Timestamp(date.year, start_month, 1)

def add_months(ts, n):
    """Add n months to a date, always return the 1st of the resulting month."""
    month = ts.month - 1 + n
    return pd.Timestamp(ts.year + month // 12, month % 12 + 1, 1)

def add_quarters(ts, n):
    """Add n quarters to a date by converting to months and using the previous func."""
    return add_months(ts, n * 3)

In [224]:
def make_delivery_periods(date):
    """
    For a given trading day 'date' build all synthetic contract delivery periods [tau_b, tau_e]
    (in decimal years) that we want to extract from the forwrad curve. 
    Return a dict of the (tau_b, tau_e) for each synthetic contract on 'date'.
    """
    periods = {}

    # -- Weekly (W1–W4) --------------------------------------
    w_start = next_monday(date)
    for k in range(1, 5):
        wb = w_start + pd.Timedelta(weeks=k - 1)
        we = wb + pd.Timedelta(weeks=1)
        periods[f"W{k}"] = (to_years(wb), to_years(we))

    # -- Monthly (M1–M6) ------------------------------------
    m_start = first_of_next_month(date)
    for k in range(1, 7):
        mb = add_months(m_start, k - 1)
        me = add_months(m_start, k)
        periods[f"M{k}"] = (to_years(mb), to_years(me))

    # -- Quarterly (Q1–Q8) ----------------------------------
    q_start = first_of_next_quarter(date)
    for k in range(1, 9):
        qb = add_quarters(q_start, k - 1)
        qe = add_quarters(q_start, k)
        periods[f"Q{k}"] = (to_years(qb), to_years(qe))

    # -- Yearly (Y1–Y3) -------------------------------------
    y_start = pd.Timestamp(date.year + 1, 1, 1)
    for k in range(1, 4):
        yb = pd.Timestamp(y_start.year + k - 1, 1, 1)
        ye = pd.Timestamp(y_start.year + k, 1, 1)
        periods[f"Y{k}"] = (to_years(yb), to_years(ye))

    return periods

In [225]:
def extract_panels(curve_store_dict, seasonal_func):
    """
    Extract synthetic price panels for all fitted trading days.
    Returns panel_W, panel_M, panel_Q, panel_Y as DataFrames with date index.
    """
    w_rows, m_rows, q_rows, y_rows = [], [], [], []

    for date in sorted(curve_store_dict.keys()):
        cs = curve_store_dict[date]
        periods = make_delivery_periods(date)

        def price_for(label, _cs=cs, _sf=seasonal_func):
            tb, te = periods[label]
            if tb < _cs["knots"][0] - 1e-6 or te > _cs["knots"][-1] + 1e-6:
                return np.nan
            return synthetic_futures_price(_cs["coeffs"], _cs["knots"], tb, te, seasonal_func=_sf)

        if p3_start <= date < p3_end:
            w_rows.append({"date": date, **{f"W{k}": price_for(f"W{k}") for k in range(1, 5)}})
        m_rows.append({"date": date, **{f"M{k}": price_for(f"M{k}") for k in range(1, 7)}})
        q_rows.append({"date": date, **{f"Q{k}": price_for(f"Q{k}") for k in range(1, 9)}})
        y_rows.append({"date": date, **{f"Y{k}": price_for(f"Y{k}") for k in range(1, 4)}})

    def to_df(rows, cols):
        return pd.DataFrame(rows).set_index("date")[cols]

    panel_W = to_df(w_rows, [f"W{k}" for k in range(1, 5)])
    panel_M = to_df(m_rows, [f"M{k}" for k in range(1, 7)])
    panel_Q = to_df(q_rows, [f"Q{k}" for k in range(1, 9)])
    panel_Y = to_df(y_rows, [f"Y{k}" for k in range(1, 4)])
    return panel_W, panel_M, panel_Q, panel_Y

In [226]:
def make_synthetic_panel(spec, contract_type, start=None, end=None):
    """
    Return synthetic price DataFrame for one spec/contract combination,
    optionally filtered to [start, end). Omit both for the full sample.
    spec: "trig", "constant", or "none"
    contract_type: "W", "M", "Q", or "Y"
    start, end: date strings or Timestamps, e.g. "2007-01-01"
    """
    panel_W, panel_M, panel_Q, panel_Y = extract_panels(curve_stores[spec], specs[spec])
    df = {"W": panel_W, "M": panel_M, "Q": panel_Q, "Y": panel_Y}[contract_type]
    if start is not None:
        df = df.loc[df.index >= start]
    if end is not None:
        df = df.loc[df.index < end]
    return df

In [227]:
M_cols = ["M1","M2","M3","M4","M5","M6"]
M_cols_W = ["M2","M3","M4","M5","M6"]
Q_cols = ["Q3","Q4","Q5","Q6","Q7","Q8"]
W_cols = ["W1","W2","W3","W4"]

periods = {
    k: ({} if t0 is None else {"start": t0, "end": t1})
    for k, (t0, t1) in period_bounds.items()
}

# Individual panels: panels[spec][period][contract_type]
panels = {
    spec: {
        period: {
            ct: make_synthetic_panel(spec, ct, **kwargs)
            for ct in ["W", "M", "Q", "Y"]
        }
        for period, kwargs in periods.items()
    }
    for spec in ["trig", "constant"]
}

# Full market (non-overlapping term structure): mkt[spec][period]
mkt = {}
for spec in ["trig", "constant"]:
    mkt[spec] = {}
    for period in ["full", "p1", "p2"]:
        mkt[spec][period] = pd.concat([
            panels[spec][period]["M"][M_cols],
            panels[spec][period]["Q"][Q_cols],
            panels[spec][period]["Y"][["Y3"]],
        ], axis=1)
    mkt[spec]["p3"] = pd.concat([
        panels[spec]["p3"]["W"][W_cols],
        panels[spec]["p3"]["M"][M_cols_W],
        panels[spec]["p3"]["Q"][Q_cols],
        panels[spec]["p3"]["Y"][["Y3"]],
    ], axis=1)

print("Done.")
print(f"  panels['trig']['full']['M']:  {panels['trig']['full']['M'].shape}")
print(f"  panels['trig']['p3']['W']:    {panels['trig']['p3']['W'].shape}")
print(f"  mkt['trig']['full']:          {mkt['trig']['full'].shape}")
print(f"  mkt['constant']['p3']:        {mkt['constant']['p3'].shape}")

Done.
  panels['trig']['full']['M']:  (5723, 6)
  panels['trig']['p3']['W']:    (1279, 4)
  mkt['trig']['full']:          (5723, 13)
  mkt['constant']['p3']:        (1279, 16)


Check synthetic price panels for negative values

In [228]:
neg_rows = []

# For both seasonalities, loop through all panels and flag any negative prices
for spec in ["trig", "constant"]:
    for ct in ["W", "M", "Q", "Y"]:
        # Use full panel to catch all dates
        df = panels[spec]["full"][ct]
        mask = df < 0
        if not mask.any().any():
            continue
        for col in df.columns[mask.any()]:
            bad = df.loc[mask[col], col]
            for date, val in bad.items():
                neg_rows.append({"spec": spec, "contract": col, "date": date.date(), "price": round(val, 4),})

# Print out the negative prices if found
if neg_rows:
    import pandas as pd
    neg_df = (pd.DataFrame(neg_rows).sort_values(["spec", "contract", "date"]).reset_index(drop=True))
    print(f"Found {len(neg_df)} negative synthetic prices:\n")
    print(neg_df.to_string(index=False))
else:
    print("No negative synthetic prices found across any spec / contract type.")

No negative synthetic prices found across any spec / contract type.


Export panels

In [ ]:
if export_constructed_panels:
    for spec in ["trig", "constant"]:
        out_path = DATA_DIR + f"panels_{spec}.xlsx"
        with pd.ExcelWriter(out_path, engine="openpyxl") as writer:

            # Individual contract type panels per period
            for period, period_cts in panels[spec].items():
                for ct, df in period_cts.items():
                    if not df.empty:
                        df.to_excel(writer, sheet_name=f"{period}_{ct}")

            # Full mkt panels per period
            for period, df in mkt[spec].items():
                if not df.empty:
                    df.to_excel(writer, sheet_name=f"mkt_{period}")

        print(f"Exported {out_path}")

### Panel visualizations

In [ ]:
# Specifications
spec_vis = "trig"  # change to "constant" to compare
plot_order = [("Y", "Y1"), ("Q", "Q1"), ("M", "M1"), ("W", "W1")]
zorder_map = {"Y1": 4, "Q1": 3, "M1": 2, "W1": 10}
linewidth_map = {"Y1": 1.0, "Q1": 1.0, "M1": 1.0, "W1": 1.0}
legend_order = ["W1", "M1", "Q1", "Y1"]
_period_list = ["full", "p1", "p2", "p3"]
_year_intervals = {"full": 4, "p1": 1, "p2": 2, "p3": 1}

# Plot near curve contrats
def plot_near_curves(spec=spec_vis, year_interval=1):
    """Plot W1/M1/Q1/Y1 near-curve contracts for all periods in one figure (1 row per period)."""
    fig, axes = plt.subplots(len(_period_list), 1, figsize=(10, 4 * len(_period_list)), sharex=False)
    for ax, period in zip(axes, _period_list):
        lines = {}
        for seg_key, col in plot_order:
            if seg_key == "W" and period != "p3":
                continue
            df = panels[spec][period][seg_key]
            if col not in df.columns:
                continue
            s = df[col].dropna()
            if s.empty:
                continue
            line, = ax.plot(s.index, s.values, lw=linewidth_map[col], label=col, zorder=zorder_map[col])
            lines[col] = line
        ax.set_title(period_labels[period], fontsize=12)
        ax.set_ylabel("EUR/MWh", fontsize=11)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        ax.xaxis.set_major_locator(mdates.YearLocator(_year_intervals.get(period, year_interval)))
        ax.tick_params(axis="both", labelsize=11)
        ax.grid(True, alpha=0.25)
        handles = [lines[k] for k in legend_order if k in lines]
        ax.legend(handles, [h.get_label() for h in handles], fontsize=11)
    fig.suptitle(f"Near-curve contracts ({spec})", fontsize=14)
    plt.tight_layout()
    if export_panel_visualizations:
        fname = RESULTS_DIR + f"near_curves_{spec}.png"
        plt.savefig(fname, dpi=150, bbox_inches="tight")
        print(f"Exported: {fname}")
    plt.close()

# Plot all contracts of one type for all periods
def plot_panel_all_contracts(ct, spec=spec_vis, year_interval=1):
    """Plot all contracts of one type for all periods in one figure (1 row per period)."""
    periods_to_plot = ["p3"] if ct == "W" else _period_list
    fig, axes = plt.subplots(len(periods_to_plot), 1, figsize=(10, 4 * len(periods_to_plot)), sharex=False)
    if len(periods_to_plot) == 1:
        axes = [axes]
    for ax, period in zip(axes, periods_to_plot):
        df = panels[spec][period][ct].dropna(how="all")
        for col in df.columns:
            s = df[col].dropna()
            ax.plot(s.index, s.values, lw=1.0, label=col)
        ax.set_title(period_labels[period], fontsize=12)
        ax.set_ylabel("EUR/MWh", fontsize=11)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        ax.xaxis.set_major_locator(mdates.YearLocator(_year_intervals.get(period, year_interval)))
        ax.tick_params(axis="both", labelsize=11)
        ax.legend(fontsize=11, ncol=2)
        ax.grid(True, alpha=0.25)
    fig.suptitle(f"{ct} contracts — all periods ({spec})", fontsize=14)
    plt.tight_layout()
    if export_panel_visualizations:
        fname = RESULTS_DIR + f"panel_{ct}_{spec}.png"
        plt.savefig(fname, dpi=150, bbox_inches="tight")
        print(f"Exported: {fname}")
    plt.close()

In [ ]:
# Full term structure one figure per contract type and spec combo
for spec in ["trig", "constant"]:
    ct_list = ["W", "M", "Q", "Y"]
    for ct in ct_list:
        plot_panel_all_contracts(ct, spec=spec)

# Near-curve W1/M1/Q1/Y1 one figure per spec
for spec in ["trig", "constant"]:
    plot_near_curves(spec=spec)

Exported: Results/panel_W_trig.png
Exported: Results/panel_M_trig.png
Exported: Results/panel_Q_trig.png
Exported: Results/panel_Y_trig.png
Exported: Results/panel_W_constant.png
Exported: Results/panel_M_constant.png
Exported: Results/panel_Q_constant.png
Exported: Results/panel_Y_constant.png
Exported: Results/near_curves_trig.png
Exported: Results/near_curves_constant.png


# 4. Descriptive Statistics

### Basic Descriptive Stats Tables

In [ ]:
def panel_stats(df_panel):
    """
    Compute descriptive statistics for price levels and log-returns.
    Returns two DataFrames: one for levels, one for log-returns.
    """
    # Levels
    level_rows = []
    for col in df_panel.columns:
        s = df_panel[col].dropna()
        if len(s) < 10:
            continue
        level_rows.append({
            "Contract": col,
            "N": len(s),
            "Mean": s.mean(),
            "Std": s.std(),
            "Min": s.min(),
            "Median": s.median(),
            "Max": s.max(),
            "Skewness": sp_stats.skew(s),
            "Kurtosis": sp_stats.kurtosis(s),   # excess kurtosis
        })
    df_levels = pd.DataFrame(level_rows).set_index("Contract")

    # Log returns
    log_ret = np.log(df_panel).diff().dropna(how="all")
    ret_rows = []
    for col in log_ret.columns:
        r = log_ret[col].dropna()
        if len(r) < 10:
            continue
        ret_rows.append({
            "Contract": col,
            "N": len(r),
            "Mean x 10^3": r.mean() * 1e3,
            "Vol (ann %)": r.std() * np.sqrt(252) * 100,
            "Skewness": sp_stats.skew(r),
            "Exc. Kurt.": sp_stats.kurtosis(r),
            "Min": r.min(),
            "Max": r.max(),
        })
    df_returns = pd.DataFrame(ret_rows).set_index("Contract")

    return df_levels, df_returns

In [ ]:
stats_results = {}

# Loop thru all seasonality spec - period - contract type combinations to compute desc stats using the funcs defed above
for spec in ["trig", "constant"]:
    stats_results[spec] = {}
    for period in period_labels:
        stats_results[spec][period] = {}
        ct_list = ["W", "M", "Q", "Y"] if period == "p3" else ["M", "Q", "Y"]
        for ct in ct_list:
            df = panels[spec][period][ct].dropna(how="all")
            if not df.empty:
                stats_results[spec][period][ct] = panel_stats(df)
        # Market model panel
        df_mkt = mkt[spec][period].dropna(how="all")
        if not df_mkt.empty:
            stats_results[spec][period]["mkt"] = panel_stats(df_mkt)

Export the results to Excel

In [ ]:
if export_desc_stats:
    for spec in ["trig", "constant"]:
        out_path = RESULTS_DIR + f"descriptive_statistics_{spec}.xlsx"
        with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
            for period, period_segs in stats_results[spec].items():
                if not period_segs:
                    continue
                startrow = 0
                for ct, (df_lev, df_ret) in period_segs.items():
                    if df_lev.empty:
                        continue
                    pd.DataFrame([[f"{ct} — Price Levels"]]).to_excel(
                        writer, sheet_name=period, startrow=startrow, index=False, header=False)
                    startrow += 1
                    df_lev.round(3).to_excel(writer, sheet_name=period, startrow=startrow)
                    startrow += len(df_lev) + 2
                    pd.DataFrame([[f"{ct} — Log Returns"]]).to_excel(
                        writer, sheet_name=period, startrow=startrow, index=False, header=False)
                    startrow += 1
                    df_ret.round(3).to_excel(writer, sheet_name=period, startrow=startrow)
                    startrow += len(df_ret) + 3
        print(f"Exported {out_path}")

Exported Results/descriptive_statistics_trig.xlsx
Exported Results/descriptive_statistics_constant.xlsx


### Seasonal Volatility Tables

In [ ]:
MONTH_NAMES = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

def seasonal_vol_table(df_panel, ann_factor=250):
    """
    Seasonal volatility table for one panel (one contract type, one period).
    Returns a DataFrame: rows = contracts, cols = [Const. vol, Jan-Dec].
    Annualized with sqrt(ann_factor).
    """
    log_ret = np.log(df_panel / df_panel.shift(1)).dropna(how="all")
    rows = []
    for col in df_panel.columns:
        r = log_ret[col].dropna()
        if len(r) < 10:
            continue
        row = {"Contract": col, "Const. vol": r.std() * np.sqrt(ann_factor)}
        for m, name in enumerate(MONTH_NAMES, 1):
            rm = r[r.index.month == m]
            row[name] = rm.std() * np.sqrt(ann_factor) if len(rm) >= 5 else np.nan
        rows.append(row)
    return pd.DataFrame(rows).set_index("Contract") if rows else pd.DataFrame()

# Compute monthly seasonal volatility for all spec - period - contract type combinations using the above func
vol_results = {}
for spec in ["trig", "constant"]:
    vol_results[spec] = {}
    for period in period_labels:
        ct_list = ["W", "M", "Q", "Y"] if period == "p3" else ["M", "Q", "Y"]
        frames = []
        for ct in ct_list:
            df_p = panels[spec][period].get(ct)
            if df_p is None or df_p.dropna(how="all").empty:
                continue
            frames.append(seasonal_vol_table(df_p))
        vol_results[spec][period] = pd.concat(frames) if frames else pd.DataFrame()

In [ ]:
# Mean absolute difference in seasonal vol - trig vs constant
_month_cols = ["Const. vol"] + MONTH_NAMES

df_tr = vol_results["trig"]["full"]
df_co = vol_results["constant"]["full"]

common_idx  = df_tr.index.intersection(df_co.index)
common_cols = [c for c in _month_cols if c in df_tr.columns and c in df_co.columns]

diff = (df_tr.loc[common_idx, common_cols] - df_co.loc[common_idx, common_cols]).abs()

mean_row = diff.mean().rename("Mean")
diff_with_mean = pd.concat([diff, mean_row.to_frame().T])

In [ ]:
if export_desc_stats:
    # Export the seasonal volatility tables for each seaosnality and period
    for spec in ["trig", "constant"]:
        out_path = RESULTS_DIR + f"seasonal_volatility_{spec}.xlsx"
        with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
            for period, df_vol in vol_results[spec].items():
                if df_vol.empty:
                    continue
                (df_vol * 100).round(1).to_excel(writer, sheet_name=period)
        print(f"Exported: {out_path}")
    
    # Export the difference table with the mean row included
    out = RESULTS_DIR + "vol_diff_trig_vs_constant.xlsx"
    (diff_with_mean * 100).round(2).to_excel(out)
    print(f"Exported: {out}")

Exported: Results/seasonal_volatility_trig.xlsx
Exported: Results/seasonal_volatility_constant.xlsx
Exported: Results/vol_diff_trig_vs_constant.xlsx


# 5. PCA

### Running PCA

In [ ]:
def log_returns(df):
    """Daily log returns ln(F_c(t)/F_c(t-1))"""
    return np.log(df / df.shift(1)).dropna(how="all")

def monthly_normalise(ret_df):
    """
    Normalise each return series by calendar month:
    subtract the monthly mean and divide by the monthly std
    """
    out = ret_df.copy().astype(float)
    for m in range(1, 13):
        mask = ret_df.index.month == m
        if mask.sum() == 0:
            continue
        monthly = ret_df.loc[mask]
        mn = monthly.mean()
        std = monthly.std().replace(0, np.nan)
        out.loc[mask] = (monthly - mn) / std
    return out

In [ ]:
def pca_segment(panel_df, exclude_cols=None):
    """
    PCA on one panel (one spec, one period, one contract type or full mkt).
    Returns dict or None if too few observations.
    """
    # get log ret
    ret = log_returns(panel_df)

    # Drop cols missing >50% to removes W contracts from mkt panel for periods where they don't yet exist
    if exclude_cols:
        ret = ret.drop(columns=[c for c in exclude_cols if c in ret.columns])
    ret = ret.loc[:, ret.isna().mean() < 0.5]  
    ret = ret.dropna(how="any")
    if len(ret) < 20 or ret.shape[1] == 0:
        return None

    # Monthly normalisation to remove calendar effects
    norm = monthly_normalise(ret).dropna(how="any")
    if len(norm) < 20:
        return None

    # Eigendecomposition of the covar matrix
    contracts = norm.columns.tolist()
    cov_mat = norm.cov().values
    evals, evecs = np.linalg.eigh(cov_mat)

    # Sort descending by eigenvalue
    order = np.argsort(evals)[::-1]
    evals = evals[order]
    evecs = evecs[:, order]

    # Variance explained by each PC and cumulative variance explained
    total = evals.sum()
    var_exp = evals / total * 100
    cum_var = np.cumsum(var_exp)

    return dict(
        contracts = contracts,
        n_obs= len(norm),
        eigenvalues = evals,
        loadings = evecs,      # column k = factor k loading vector
        var_exp = var_exp,
        cum_var = cum_var,
        corr = norm.corr(),
    )

In [ ]:
# PCA configuration - contract segments to analyse and number of factors to retain
SEGMENTS = {
    "W" : ("W", []),
    "M": ("M", []),
    "Q" : ("Q", []),
    "Y" : ("Y", []),
    "mkt": ("mkt", []),
}
N_FACTORS = 3

def _enough_coverage(df_seg, ref_df, min_frac=0.5):
    """Check whether a segment has sufficient data coverage to run PCA on.
    Compares the number of non-empty rows in df_seg against ref_df.
    Returns True if df_seg covers at least min_frac of ref_df's trading days.
    """
    n_seg = df_seg.dropna(how="all").shape[0]
    n_ref = ref_df.dropna(how="all").shape[0]
    return (n_ref > 0) and (n_seg / n_ref) >= min_frac

pca_results = {}

# Loop through all spec - period - segment combinations and run PCA where coverage is sufficient
for spec in ["trig", "constant"]:
    pca_results[spec] = {}
    for period in period_labels:
        pca_results[spec][period] = {}
        ref_df = panels[spec][period]["M"]

        for seg_name, (ct, excl) in SEGMENTS.items():
            # Skip W segments for periods where W contracts don't exist
            if ct == "W" and period != "p3":
                continue
            
            # For the mkt segment, use the full mkt panel
            if ct == "mkt":
                # For p3 use full mkt with W contracts, for other periods use mkt panel without W contracts
                if period == "p3":
                    df_seg = pd.concat([
                        panels[spec]["p3"]["M"][M_cols],
                        panels[spec]["p3"]["Q"][Q_cols],
                        panels[spec]["p3"]["Y"][["Y3"]],
                    ], axis=1)
                else:
                    df_seg = mkt[spec][period]
            else:
                df_seg = panels[spec][period][ct]

            # Check coverage
            if not _enough_coverage(df_seg, ref_df):
                pca_results[spec][period][seg_name] = None
                continue
            
            # Run PCA on the segment
            res = pca_segment(df_seg, exclude_cols=excl)
            pca_results[spec][period][seg_name] = res

            if res is None:
                continue

        # Extra for p3 - mkt with W contracts
        if period == "p3":
            df_mkt_w = mkt[spec]["p3"]
            if _enough_coverage(df_mkt_w, ref_df):
                res_w = pca_segment(df_mkt_w)
                pca_results[spec]["p3"]["mkt_W"] = res_w
            else:
                pca_results[spec]["p3"]["mkt_W"] = None

Export PCA results

In [ ]:
if export_pca_results:
    for spec in ["trig", "constant"]:
        
        # File 1:Factor loadings + variance explained
        out_path = RESULTS_DIR + f"pca_results_{spec}.xlsx"
        sheets_written = 0
        with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
            for period, desc in period_labels.items():
                for seg_name, res in pca_results[spec][period].items():
                    if res is None:
                        continue
                    sheet = f"{period}_{seg_name}"[:31]
                    startrow = 0
                    meta = pd.DataFrame([
                        ["Spec", spec], ["Period", desc],
                        ["Segment", seg_name], ["N obs", res["n_obs"]],
                        ["Contracts", ", ".join(res["contracts"])],
                    ])
                    meta.to_excel(writer, sheet_name=sheet, startrow=startrow, index=False, header=False)
                    startrow += len(meta) + 1
                    C = len(res["contracts"])
                    var_df = pd.DataFrame({
                        "Factor": range(1, C + 1),
                        "Eigenvalue": res["eigenvalues"],
                        "Var Explained (%)": res["var_exp"],
                        "Cum Var (%)": res["cum_var"],
                    }).set_index("Factor")
                    pd.DataFrame([["Variance Explained"]]).to_excel(
                        writer, sheet_name=sheet, startrow=startrow, index=False, header=False)
                    startrow += 1
                    var_df.round(3).to_excel(writer, sheet_name=sheet, startrow=startrow)
                    startrow += len(var_df) + 2
                    n_f = min(N_FACTORS, C)
                    loadings_df = pd.DataFrame(
                        res["loadings"][:, :n_f],
                        index=res["contracts"],
                        columns=[f"Factor {i+1}" for i in range(n_f)]
                    )
                    pd.DataFrame([["Factor Loadings"]]).to_excel(
                        writer, sheet_name=sheet, startrow=startrow, index=False, header=False)
                    startrow += 1
                    loadings_df.round(4).to_excel(writer, sheet_name=sheet, startrow=startrow)
                    sheets_written += 1
        print(f"Exported: {out_path} ({sheets_written} sheets)")

        # File 2:Correl matrices
        out_corr = RESULTS_DIR + f"pca_correlations_{spec}.xlsx"
        sheets_corr = 0
        with pd.ExcelWriter(out_corr, engine="openpyxl") as writer:
            for period, desc in period_labels.items():
                for seg_name, res in pca_results[spec][period].items():
                    if res is None or "corr" not in res:
                        continue
                    sheet = f"{period}_{seg_name}"[:31]
                    pd.DataFrame([[f"{spec} | {desc} | {seg_name}"]]).to_excel(
                        writer, sheet_name=sheet, index=False, header=False)
                    res["corr"].round(4).to_excel(writer, sheet_name=sheet, startrow=2)
                    sheets_corr += 1
        print(f"Exported: {out_corr} ({sheets_corr} sheets)")

Exported: Results/pca_results_trig.xlsx (18 sheets)
Exported: Results/pca_correlations_trig.xlsx (18 sheets)
Exported: Results/pca_results_constant.xlsx (18 sheets)
Exported: Results/pca_correlations_constant.xlsx (18 sheets)


### PCA Plots

In [ ]:
# Plot settings
spec_colors = {"trig": "darkblue", "constant": "red"}
spec_ls = {"trig": "-",         "constant": "--"}
spec_labels_pca = {"trig": "Trigonometric", "constant": "Constant"}

PAIRS = [["M", "Q"], ["Y", "mkt"]]

# Plot PCA loadings for the first N_FACTORS factors  for each segment and period comparing the 2 seasonalities 
for period, desc in period_labels.items():
    height_ratios = [1] * N_FACTORS + [0.12] + [1] * N_FACTORS
    gs = gridspec.GridSpec(
        N_FACTORS * 2 + 1, 2,
        height_ratios=height_ratios,
        hspace=0.3, wspace=0.08,
        top=0.93, bottom=0.05, left=0.08, right=0.97,
    )
    fig = plt.figure(figsize=(12, 2.6 * N_FACTORS * 2))
    
    for pair_idx, pair in enumerate(PAIRS):
        row_offset = pair_idx * (N_FACTORS + 1)
        for col_idx, seg_name in enumerate(pair):
            for f in range(N_FACTORS):
                ax = fig.add_subplot(gs[row_offset + f, col_idx])

                any_data = False
                title_parts = []
                for spec in ["trig", "constant"]:
                    res = pca_results[spec][period].get(seg_name)
                    if res is None or f >= res["loadings"].shape[1]:
                        continue
                    contracts = res["contracts"]
                    x = np.arange(len(contracts))
                    loading = res["loadings"][:, f].copy()
                    if loading[np.argmax(np.abs(loading))] < 0:
                        loading = -loading
                    ax.plot(x, loading, marker="o", ms=4, lw=1.2,
                            color=spec_colors[spec], ls=spec_ls[spec],
                            label=spec_labels_pca[spec])
                    ax.set_xticks(x)
                    ax.set_xticklabels(contracts, fontsize=15, rotation=0)
                    if f == 0:
                        title_parts.append(f"{spec_labels_pca[spec]}: F1={res['var_exp'][0]:.0f}%")
                    any_data = True

                ax.axhline(0, color="k", lw=0.6, ls="--")
                ax.set_ylim(-1.1, 1.1)
                ax.set_yticks([-1.0, -0.5, 0.0, 0.5, 1.0])
                ax.grid(True, alpha=0.2)

                if col_idx != 0:
                    ax.tick_params(labelleft=False)
                else:
                    ax.tick_params(axis="y", labelsize=15)

                if col_idx == 0:
                    ax.set_ylabel(f"Factor {f+1}", fontsize=17)
                if pair_idx == 0 and f == 0 and col_idx == 1 and any_data:
                    ax.legend(fontsize=13, loc="lower right")

    # Ezport the plots
    if export_pca_results:
        plt.savefig(RESULTS_DIR + f"pca_loadings_{period}.png", dpi=150, bbox_inches="tight")
        print(f"Exported: {RESULTS_DIR}pca_loadings_{period}.png")
        plt.close()

Exported: Results/pca_loadings_full.png
Exported: Results/pca_loadings_p1.png
Exported: Results/pca_loadings_p2.png
Exported: Results/pca_loadings_p3.png


In [ ]:
# Plot extra figs for p3 - W (left) and mkt with W (right)
def _plot_seg(ax, seg_name, f, col_idx, first_ax):
    any_data = False
    for spec in ["trig", "constant"]:
        res = pca_results[spec]["p3"].get(seg_name)
        if res is None or f >= res["loadings"].shape[1]:
            continue
        contracts = res["contracts"]
        x = np.arange(len(contracts))
        loading = res["loadings"][:, f].copy()
        if loading[np.argmax(np.abs(loading))] < 0:
            loading = -loading
        ax.plot(x, loading, marker="o", ms=4, lw=1.2,
                color=spec_colors[spec], ls=spec_ls[spec],
                label=spec_labels_pca[spec])
        ax.set_xticks(x)
        ax.set_xticklabels(contracts, fontsize=15, rotation=0)
        any_data = True

    ax.axhline(0, color="k", lw=0.6, ls="--")
    ax.set_ylim(-1.1, 1.1)
    ax.set_yticks([-1.0, -0.5, 0.0, 0.5, 1.0])
    ax.grid(True, alpha=0.2)

    if col_idx != 0:
        ax.tick_params(labelleft=False)
    else:
        ax.tick_params(axis="y", labelsize=15)
        ax.set_ylabel(f"Factor {f+1}", fontsize=17)

    if first_ax and any_data:
        ax.legend(fontsize=13, loc="lower right")

gs_wm = gridspec.GridSpec(
    N_FACTORS, 2,
    hspace=0.3, wspace=0.08,
    top=0.93, bottom=0.05, left=0.08, right=0.97,
)
fig_wm = plt.figure(figsize=(12, 1.3 * N_FACTORS * 2))

for f in range(N_FACTORS):
    for col_idx, seg_name in enumerate(["W", "mkt_W"]):
        ax = fig_wm.add_subplot(gs_wm[f, col_idx])
        _plot_seg(ax, seg_name, f, col_idx, first_ax=(f == 0 and col_idx == 0))

# Export the W and mkt_W plots for p3
if export_pca_results:
    plt.savefig(RESULTS_DIR + "pca_loadings_p3_W_mktW.png", dpi=150, bbox_inches="tight")
    print(f"Exported: {RESULTS_DIR}pca_loadings_p3_W_mktW.png")
    plt.close()

Exported: Results/pca_loadings_p3_W_mktW.png


# 6. Parametric Market Model


NB: This part is done only for the trigonometric seasonality specification and for the Full Sample period.

### Define the mkt model functions

In [256]:
# ---- Seasonal variance  ------------------------------
def _seasonal_design(dates, L=4):
    t = dates.dayofyear.values.astype(float)
    cols = [np.ones(len(t))]
    for l in range(1, L + 1):
        cols.append(np.cos(2 * np.pi * l * t / 365))
        cols.append(np.sin(2 * np.pi * l * t / 365))
    return np.column_stack(cols)

def fit_seasonal_vol(log_ret_df, L=4):
    """OLS fit of sigma^2_S(t) = c1 + sum[c_2l*cos + c_2l+1*sin].
    Standardises each contract by its own mean and std, then avg
    squared standardised returns."""
    eps = (log_ret_df - log_ret_df.mean()) / log_ret_df.std()
    sq_mean = eps.pow(2).mean(axis=1).dropna()
    dates = sq_mean.index
    X = _seasonal_design(dates, L=L)
    coef, *_ = np.linalg.lstsq(X, sq_mean.values, rcond=None)
    sigma2_s = pd.Series(X @ coef, index=dates).clip(lower=1e-10)
    return coef, np.sqrt(sigma2_s)

# ---- Maturity volatility  -----------------------------
def _ns_vol(tau, sigma0, sigma1, sigma2, kappa):
    """Nelson-Siegel: sigma_tilde(tau) = sigma0 + (sigma1 + sigma2*tau)*exp(-kappa*tau)."""
    return sigma0 + (sigma1 + sigma2 * tau) * np.exp(-kappa * tau)

In [257]:
def fit_parametric_model(log_ret_df, sigma_s, tau_dict, n_factors=3):
    """
    Normalise returns by sigma_S(t) after subtracting contract mean A_c,
    run PCA, and fit Nelson-Siegel maturity vol curves for top n_factors.
    tau_dict: {column_name -> time-to-start-of-delivery in years}
    """
    # Returns normalisation
    A_c = log_ret_df.mean()               
    sig = sigma_s.shift(1).reindex(log_ret_df.index).dropna()
    lr_n = (log_ret_df.loc[sig.index].sub(A_c, axis=1).div(sig, axis=0).dropna(how="all"))
    lr_n = lr_n.dropna(axis=1, thresh=max(10, int(0.3 * len(lr_n))))

    # PCA on normalised returns
    cov = lr_n.cov().values
    eigvals, eigvecs = np.linalg.eigh(cov)
    idx = np.argsort(eigvals)[::-1]
    eigvals, eigvecs = eigvals[idx], eigvecs[:, idx]
    eigvals = np.maximum(eigvals, 0.0)

    # Variance explained by each PC
    total_var = eigvals.sum()
    var_exp = eigvals / total_var if total_var > 0 else np.zeros_like(eigvals)

    # Fit Nelson-Siegel to loadings of the top n_factors PCs
    contracts = lr_n.columns.tolist()
    C = len(contracts)
    tau_arr = np.array([tau_dict[c] for c in contracts], dtype=float)

    # Store fitted params and fitted curves for each factor, along with empirical volatilities (PC loadings)
    mat_params, mat_fitted, emp_vols = {}, {}, {}
    for k in range(min(n_factors, C)):
        ev = np.sqrt(eigvals[k]) * eigvecs[:, k]
        emp_vols[k + 1] = ev
        
        try:
            abs_mean = float(np.abs(ev).mean())
            _lb = [-1.0, -1.0, -5.0, 0.01]
            _ub = [ 1.0,  1.0,  5.0, 20.0]
            p0 = [np.clip(v, lo, hi) for v, lo, hi in zip(
                [max(abs_mean, 1e-6), float(ev[0]) - abs_mean, 0.0, 1.0], _lb, _ub)]
            popt, _ = curve_fit(_ns_vol, tau_arr, ev, p0=p0, bounds=([-1.0, -1.0, -5.0, 0.01],
                                                                    [ 1.0,  1.0,  5.0, 20.0]),maxfev=8000)
        except RuntimeError:
            popt = [np.nan] * 4
        
        mat_params[k + 1] = dict(zip(["sigma0", "sigma1", "sigma2", "kappa"], popt))
        mat_fitted[k + 1] = _ns_vol(tau_arr, *popt)

    # Return all PCA outputs - eigendecomposition, variance explained, NS fits, normalised returns
    return dict(
        eigenvalues=eigvals, eigenvectors=eigvecs, var_explained=var_exp, contracts=contracts,
        tau=tau_arr, emp_vols=emp_vols, maturity_params=mat_params, maturity_fitted=mat_fitted,
        lr_norm=lr_n, cov=cov,
    )

### Select optimal number of Fourier harmonics L

In [ ]:
# Fit L = 1 - 6 and compare BIC.
L_RANGE = range(1, 7)
ct_titles = {"M": "Monthly", "Q": "Quarterly", "Y": "Yearly"}

# Compute AIC and BIC for the seasonal variance model with L harmonics
def _aic_bic(log_ret_df, L):
    eps = (log_ret_df - log_ret_df.mean()) / log_ret_df.std()
    sq_mean = eps.pow(2).mean(axis=1).dropna()
    n = len(sq_mean)
    if n < 10:
        return np.nan, np.nan
    X = _seasonal_design(sq_mean.index, L=L)
    k = X.shape[1]
    b, *_ = np.linalg.lstsq(X, sq_mean.values, rcond=None)
    rss = np.sum((sq_mean.values - X @ b) ** 2)
    s2 = rss / n
    ll = -n / 2 * (np.log(2 * np.pi * s2) + 1)
    return -2 * ll + 2 * k, -2 * ll + k * np.log(n)

# Fit seasonal vol with L = 1-6 and store BIC results
spec = "trig"
period = "full"
rows = []

for ct in ["M", "Q", "Y"]:
    price_df = panels[spec][period][ct]
    if price_df.empty or price_df.dropna(how="all").shape[0] < 30:
        continue
    log_ret = np.log(price_df).diff().dropna(how="all")
    bics = {}
    for L in L_RANGE:
        _, bic = _aic_bic(log_ret, L)
        bics[L] = bic
    best_L = min(bics, key=bics.get)
    rows.append({"spec": spec, "period": period, "ct": ct, "best_L": best_L,
                 **{f"BIC_L{L}": round(bics[L], 1) for L in L_RANGE}})

df_L = pd.DataFrame(rows)

if export_mkt_model_results:
    _out = RESULTS_DIR + "mkt_model_L_selection.xlsx"
    df_L.to_excel(_out, index=False)
    print(f"Exported: {_out}")


Exported: Results/mkt_model_L_selection.xlsx


In [259]:
# Plot BIC curves for all contract types
cts_to_plot = ["M", "Q", "Y"]
fig, axes = plt.subplots(1, len(cts_to_plot), figsize=(2.55 * len(cts_to_plot), 2), sharey=False)

for ci, ct in enumerate(cts_to_plot):
    ax = axes[ci]
    row = df_L[(df_L["period"] == "full") & (df_L["ct"] == ct)]
    if row.empty:
        ax.set_visible(False)
        continue
    bics = [row.iloc[0][f"BIC_L{L}"] for L in L_RANGE]
    best = row.iloc[0]["best_L"]
    ax.plot(list(L_RANGE), bics, marker="o", ms=4, lw=1.4, color="purple")
    ax.axvline(best, color="purple", lw=1.0, ls="--", alpha=0.7, label=f"Best L={best}")
    ax.set_xticks(list(L_RANGE))
    ax.yaxis.set_major_locator(plt.MaxNLocator(nbins=3, min_n_ticks=3))
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=8)
    ax.set_title(ct_titles[ct], fontsize=10)
    ax.set_xlabel("L", fontsize=10)
    if ci == 0:
        ax.set_ylabel("BIC", fontsize=10)

plt.tight_layout()
if export_mkt_model_results:
    fname = RESULTS_DIR + "mkt_model_L_BIC.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    print(f"Exported: {fname}")
plt.close()

Exported: Results/mkt_model_L_BIC.png


### Fit market model

In [260]:
# Setup the mkt model
DELTA_YEARS = {"M": 1 / 12, "Q": 1 / 4, "Y": 1.0}
VAR_THRESHOLD = 0.7     # minimum cumulative variance to retain factors
MAX_FACTORS = 6         # hard cap (avoids over-fitting on short series)
N_FACTORS_FIXED = None  # set to int (e.g. 3) to fix n_factors globally, None = use VAR_THRESHOLD
L_SEASONAL = {"M": None, "Q": 2, "Y": None} # Set each to an int to fix L for that segment, or None to use BIC-optimal.

# Get L for each contract type using BIC results or specified values
def _get_L(spec, period, ct):
    val = L_SEASONAL.get(ct)
    if val is not None:
        return val
    if "df_L" in globals() and not df_L.empty:
        row = df_L[(df_L["spec"]==spec) & (df_L["period"]==period) & (df_L["ct"]==ct)]
        if not row.empty:
            return int(row.iloc[0]["best_L"])
    return 4     # Default = 4 as per Benth et al. (2008)

In [273]:
param_results = {"trig": {"full": {}}}

# Fit the full parametric market model for all contract types
for ct in ["M", "Q", "Y"]:
    price_df = panels["trig"]["full"][ct]
    if price_df.empty or price_df.dropna(how="all").shape[0] < 30:
        continue

    # Map each contract to its time-to-delivery (years) and compute log returns
    delta = DELTA_YEARS[ct]
    tau_dict= {col: (int(col[len(ct):]) - 1) * delta for col in price_df.columns}
    log_ret = np.log(price_df).diff().dropna(how="all")
    L = _get_L("trig", "full", ct)

    try:
        coef_s, sigma_s = fit_seasonal_vol(log_ret, L=L)

        # Full PCA to determine how many factors to retain
        A_c= log_ret.mean()
        sig = sigma_s.shift(1).reindex(log_ret.index).dropna()
        lr_n = (log_ret.loc[sig.index].sub(A_c, axis=1).div(sig, axis=0).dropna(how="all"))
        lr_n = lr_n.dropna(axis=1, thresh=max(10, int(0.3 * len(lr_n))))
        C = lr_n.shape[1]

        # Determine number of factors: fixed override or minimum needed to explain VAR_THRESHOLD of variance
        cov_full = lr_n.cov().values
        evals_full, _ = np.linalg.eigh(cov_full)
        evals_full= np.maximum(evals_full[::-1], 0.0)
        cum_ve = np.cumsum(evals_full) / evals_full.sum()
        n_f = N_FACTORS_FIXED if N_FACTORS_FIXED is not None else int(np.searchsorted(cum_ve, VAR_THRESHOLD) + 1)
        n_f = min(n_f, MAX_FACTORS, C)

        # Fit the full parametric market model and store results
        res = fit_parametric_model(log_ret, sigma_s, tau_dict, n_factors=n_f)
        res["seasonal_coef"] = coef_s
        res["tau_full"] = tau_dict
        res["n_factors"] = n_f
        res["L_seasonal"] = L
        param_results["trig"]["full"][ct] = res
    except Exception as e:
        print(f"  [SKIP] {ct}: {e}")

print("Parametric model fitted:")
for ct, res in param_results["trig"]["full"].items():
    print(f"  {ct}: {res['n_factors']}f  L{res['L_seasonal']}")

Parametric model fitted:
  M: 3f  L1
  Q: 3f  L2
  Y: 1f  L1


In [262]:
# Export results to Excel
if export_mkt_model_results:
    rows_ve, rows_ns, rows_coef = [], [], []
    
    for ct in ["M", "Q", "Y"]:
        res = param_results["trig"]["full"].get(ct)
        if res is None:
            continue
        nf = res["n_factors"]
        L = res.get("L_seasonal", "?")
        ve = res["var_explained"]
        cum = np.cumsum(ve)
    
        # Variance explained - all components
        n_all = len(res["eigenvalues"])
        for k in range(n_all):
            rows_ve.append({
                "Contract": ct, "L": L, "Factor": f"PC{k+1}",
                "Retained": k < nf,
                "Eigenvalue": round(float(res["eigenvalues"][k]), 6),
                "Var explained (%)": round(float(ve[k]) * 100, 2),
                "Cumulative (%)": round(float(cum[k]) * 100, 2),
            })
    
        # Nelson-Siegel params
        for k in range(nf):
            p = res["maturity_params"].get(k + 1)
            if p is None:
                continue
            rows_ns.append({"Contract": ct, "Factor": f"PC{k+1}", **{key: round(val, 6) for key, val in p.items()}})
    
        # Seasonal coeffs
        coef = res["seasonal_coef"]
        L_c = (len(coef) - 1) // 2
        row = {"Contract": ct, "L": L_c, "beta_0": round(coef[0], 6)}
        for l in range(1, L_c + 1):
            row[f"cos_{l}"] = round(coef[2*l - 1], 6)
            row[f"sin_{l}"] = round(coef[2*l],     6)
        rows_coef.append(row)
    
    out = RESULTS_DIR + "mkt_model_results.xlsx"
    with pd.ExcelWriter(out) as writer:
        pd.DataFrame(rows_ve).to_excel(writer, sheet_name="Variance_Explained", index=False)
        pd.DataFrame(rows_ns).to_excel(writer, sheet_name="NS_Parameters", index=False)
        pd.DataFrame(rows_coef).to_excel(writer, sheet_name = "Seasonal_Coef", index=False)
    print(f"Exported: {out}")


Exported: Results/mkt_model_results.xlsx


### Seasonal and Maturity Volatility Plots

Seasonality variance

In [263]:
# Setup
_MONTH_DOYS = [1, 32, 60, 91, 121, 152, 182, 213, 244, 274, 305, 335]
t_plot = np.linspace(1, 365.25, 500)
ct_list_pm = ["M", "Q", "Y"]
ct_titles = {"M": "Monthly", "Q": "Quarterly", "Y": "Yearly"}

# Construct seasonal design matrix X for given array of day-of-year values and number of harmonics L
def _X_from_doy(t_arr, L):
    cols = [np.ones(len(t_arr))]
    for l in range(1, L + 1):
        cols.append(np.cos(2 * np.pi * l * t_arr / 365))
        cols.append(np.sin(2 * np.pi * l * t_arr / 365))
    return np.column_stack(cols)

# Plot fitted seasonal variance for each contract type
fig, axes = plt.subplots(1, len(ct_list_pm), figsize=(8, 2.3), sharey=False)
for col, ct in enumerate(ct_list_pm):
    ax  = axes[col]
    res = param_results["trig"].get( "full", {}).get(ct)
    if res is None:
        ax.set_visible(False)
        continue
    L = res.get("L_seasonal", 4)
    X_plot = _X_from_doy(t_plot, L)
    sigma2_fit = np.maximum(X_plot @ res["seasonal_coef"], 0)
    ax.plot(t_plot, sigma2_fit, color="seagreen", lw=1.5)
    ax.set_ylim(0, 1.5)
    ax.set_yticks([0, 0.5, 1.0, 1.5])
    ax.set_title(ct_titles[ct], fontsize=10)
    ax.set_xticks(_MONTH_DOYS)
    ax.set_xticklabels(list(range(1, 13)), fontsize=10)
    ax.set_xlabel("Month")
    if col == 0:
        ax.set_ylabel("Seasonal variance")

# Export the plot
plt.tight_layout()
if export_mkt_model_results:
    fname = RESULTS_DIR + "seasonal_variance.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    print(f"Exported: {fname}")
plt.close()

Exported: Results/seasonal_variance.png


Maturity volatility

In [269]:
# Setup
_MAT_COLORS = ["steelblue", "seagreen", "firebrick"]
_MAT_LS = ["--", "-.", ":"]
_MAT_MARKERS = ["^", "o", "s"]

def _plot_maturity_ax(ax, res, tau_fine=400, label_legend=True):
    """Plot empirical maturity vol scatter and fitted NS curves onto a single axes."""
    tau = res["tau"]
    tau_f = np.linspace(tau.min(), tau.max(), tau_fine)
    nf = res.get("n_factors", len(res.get("emp_vols", {})))
    overall_emp = np.zeros(len(tau))
    overall_fitted = np.zeros(len(tau_f))
    any_fitted = False

    for k in range(nf):
        if (k + 1) not in res["emp_vols"]:
            continue
        ev = res["emp_vols"][k + 1]
        col= _MAT_COLORS[k % len(_MAT_COLORS)]
        ls = _MAT_LS[k % len(_MAT_LS)]
        mk = _MAT_MARKERS[k % len(_MAT_MARKERS)]
        ax.scatter(tau, ev, s=22, color=col, marker=mk, alpha=0.85, zorder=3, label=f"PC{k+1}" if label_legend else None)
        overall_emp += ev ** 2
        p = res["maturity_params"].get(k + 1)
        if p is not None and not any(np.isnan(list(p.values()))):
            fitted = _ns_vol(tau_f, p["sigma0"], p["sigma1"], p["sigma2"], p["kappa"])
            ax.plot(tau_f, fitted, color=col, lw=1.5, ls=ls)
            overall_fitted += fitted ** 2
            any_fitted = True

    ax.scatter(tau, np.sqrt(overall_emp), s=28, color="black", marker="D", 
            alpha=0.9, zorder=4, label="Overall" if label_legend else None)
    if any_fitted:
        ax.plot(tau_f, np.sqrt(overall_fitted), color="black", lw=1.7, ls="-")
    ax.axhline(0, color="k", lw=0.5, ls=":", alpha=0.5)

In [270]:
ct_list_pm = ["M", "Q", "Y"]
ct_titles  = {"M": "Monthly", "Q": "Quarterly", "Y": "Yearly"}

# Plot fitted maturity vol curves comparing empirical vol (PC loadings) against fitted NS curves
fig, axes = plt.subplots(1, len(ct_list_pm), figsize=(2.8 * len(ct_list_pm), 3.1))
for ci, ct in enumerate(ct_list_pm):
    ax = axes[ci]
    res = param_results["trig"].get("full", {}).get(ct)
    if res is None:
        ax.set_visible(False)
        continue
    _plot_maturity_ax(ax, res)
    ax.set_title(ct_titles[ct], fontsize=10)
    ax.set_xlabel("Time to delivery (in years)", fontsize=10)
    if ci == 0:
        ax.set_ylabel("Volatility", fontsize=10)

fig.suptitle(f"Maturity volatility", fontsize=12)
handles, labels_leg = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_leg, loc="lower center", ncol=len(handles), fontsize=9, bbox_to_anchor=(0.5, -0.08), frameon=True)
plt.tight_layout()

# Export plot
if export_mkt_model_results:
    fname = RESULTS_DIR + "maturity_volatility.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    print(f"Exported: {fname}")
plt.close()

Exported: Results/maturity_volatility.png


In [271]:
_L_vals   = [1, 2, 3, 4]
_cts_comp = ["M", "Q", "Y"]

# Maturity vol comparison: 4 rows (L=1..4) x 3 cols (M, Q, Y)
fig, axes = plt.subplots(len(_L_vals), len(_cts_comp), figsize=(2.5 * len(_cts_comp), 2 * len(_L_vals)), sharey=False)

for ri, L_val in enumerate(_L_vals):
    for ci, ct in enumerate(_cts_comp):
        ax = axes[ri][ci]

        # Recompute model for each L
        price_df = panels["trig"]["full"][ct]
        delta = DELTA_YEARS[ct]
        tau_dict_c = {col: (int(col[len(ct):]) - 1) * delta for col in price_df.columns}
        log_ret = np.log(price_df).diff().dropna(how="all")
        n_f = param_results["trig"]["full"][ct]["n_factors"]
        try:
            coef_s, sigma_s = fit_seasonal_vol(log_ret, L=L_val)
            res = fit_parametric_model(log_ret, sigma_s, tau_dict_c, n_factors=n_f)
        except Exception:
            ax.set_visible(False)
            continue

        _plot_maturity_ax(ax, res, label_legend=(ri == 0))

        if ri == 0:
            ax.set_title(ct_titles[ct] + " contracts", fontsize=10)
        if ci == 0:
            ax.set_ylabel(f"L={L_val}", fontsize=10)
        if ri == len(_L_vals) - 1:
            ax.set_xlabel("Time to delivery (years)", fontsize=10)

handles, labels = axes[0][0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncol=len(handles),fontsize=9, bbox_to_anchor=(0.5, -0.02), frameon=True)
plt.tight_layout()

# Export
if export_mkt_model_results:
    fname = RESULTS_DIR + "maturity_vol_L_comparison.png"
    plt.savefig(fname, dpi=150, bbox_inches="tight")
    print(f"Exported: {fname}")
plt.close()

Exported: Results/maturity_vol_L_comparison.png
